# 43B — S6 symmetric two-crop TTA standalone

**Creates submission.csv for ONE competition submission.** This is standalone S6, NOT anchor33/39 and NOT a10% blend. Its Public score may be lower than0.958; this is an experiment to isolate crop TTA. Do not replace selected final candidates33/39 with these controls merely because they are newer.

43A uses the original centre offset16,16 and unchanged S6 worker.43B uses crop offsets8,8 and24,24, equal mean of complete study logits, then the same S6 output transform as A. Both use the same learned S6 checkpoint, anatomy, windows, crop288/cache320, fp16, chunk96 and one loader worker/GPU. Both export per-target percentile ranks (same AUC ordering as their non-saturated probabilities). No gold fitting, training, Raptor fallback or R6 models.

Attach exactly **THREE inputs**:
1. RSNA Knee competition test DICOMs.
2. [Nartaa S6 all-data weights Version5](https://www.kaggle.com/datasets/nartaa/rsna-knee-s6fall-weights/versions/5).
3. [Heliosli OrthoFoundation code/anatomy Version1](https://www.kaggle.com/datasets/heliosli/rsna-orthofoundation-fold0-epoch7-20261004/versions/1).

**No R6 bundle, publication SWA/Raptor weights, study cache or prior Output needed. T4x2, Internet Off, Save & Run All.** Use separate notebooks for43A/B. Import metadata is not proof the draft attachments are correct. Require `rsna_43B/43B_run_audit.json` status COMPLETE before submitting. Return each Public score and Output ZIP.

Budgets: eight-hour global cap, S6 worker budget up to seven hours; these are safety timeouts, not measured ETAs.43B performs two S6 forwards per study, but decodes once. A/B can run independently, but submitting both uses today's two slots. Check available GPU quota first. No GPU/hidden-run guarantee:40A's hidden exception is unresolved. Removing anchor stages removes that branch, not proof that all rerun risks are fixed.

Scientific failure policy: one retry by inherited worker; any study still unscored causes a hard error and no submission.csv. No constant fill or silent different-model fallback. This strictness keeps the comparison interpretable but can cause a submission exception for a genuinely unprocessable hidden study.

Crop changes are expected to change visible values. Original worker identity is verified before the explicit new patch, whose hash and two offsets are checked in receipts. AUC equality/difference can be hidden by rounded Public scores. A positive standalone result is not proof of an anchor gain; a null result does not prove there is no effect. Gold58 has already been reused and is not a clean validation set. Further weight fitting is not part of this test.

Credits/licence/provenance: Nartaa S6, Heliosli code/anatomy; inherit third-party terms. Code/weights have different declared licences; this is not an eligibility determination.


In [ ]:

import hashlib, importlib.util, json, os, signal, subprocess, sys, time, traceback
from pathlib import Path
import numpy as np
import pandas as pd
WORK=Path('/kaggle/working')
OUT=WORK/'rsna_43B'
OUT.mkdir(parents=True,exist_ok=True)
START=time.time()
DEADLINE=START+8*3600
EXPERIMENT='43B'
REPORT={'experiment':EXPERIMENT,'status':'STARTED','mode':'symmetric_two_crop',
        'standalone_S6':True,'uses_anchor33':False,'uses_Raptor':False,'uses_R6':False,
        'training_started':False,'gold_used':False,'weights_fitted':False,
        'per_study_model_fallback_allowed':False,'new_public_score':None,
        'output_transform':'per-target percentile rank, same transform for A/B',
        'caveat':'Public comparison of two fixed variants; not evidence of Private top10.'}

def save_report():
    REPORT['total_seconds']=round(time.time()-START,2)
    (OUT/(EXPERIMENT+'_run_audit.json')).write_text(json.dumps(REPORT,indent=2,allow_nan=False))

def clear_partial_submission():
    p=WORK/'submission.csv'
    if p.is_file():p.unlink()

def run_stage(name,script,max_seconds):
    path=OUT/(name+'.py')
    path.write_bytes(script.encode('utf-8'))
    log_path=OUT/(name+'.log')
    budget=min(max_seconds,DEADLINE-time.time()-120)
    if budget<=0:raise RuntimeError('Global eight-hour budget exhausted before '+name)
    env=dict(os.environ,RSNA39_DEADLINE=str(DEADLINE),PYTHONDONTWRITEBYTECODE='1')
    started=time.time()
    with log_path.open('w') as log:
        p=subprocess.Popen([sys.executable,'-u',str(path)],stdout=log,stderr=subprocess.STDOUT,
                           env=env,cwd=str(WORK),start_new_session=True)
        last=0
        try:
            while p.poll() is None:
                if time.time()-started>budget:raise TimeoutError(name+' exceeded stage/global budget')
                if time.time()-last>=60:
                    last=time.time()
                    tail=log_path.read_text(errors='replace')[-1000:]
                    print(name,'elapsed',round(time.time()-started),'seconds\n'+tail,flush=True)
                time.sleep(2)
            if p.returncode!=0:
                raise RuntimeError(name+' failed rc='+str(p.returncode)+'\n'+log_path.read_text(errors='replace')[-4500:])
        finally:
            # Reap any surviving DataLoader/model subprocesses, including on failure.
            # Some inherited launchers may create their own process groups.
            try:
                import psutil
                descendants=psutil.Process(p.pid).children(recursive=True)
                for child in reversed(descendants):
                    try:child.kill()
                    except psutil.NoSuchProcess:pass
                psutil.wait_procs(descendants,timeout=10)
            except (ImportError,ProcessLookupError):pass
            except Exception as cleanup_error:
                print('Cleanup diagnostic:',repr(cleanup_error),flush=True)
            try:os.killpg(p.pid,signal.SIGKILL)
            except ProcessLookupError:pass
            p.wait(timeout=30)
    REPORT[name]={'seconds':round(time.time()-started,2),'script_sha256':hashlib.sha256(script.encode()).hexdigest()}
    save_report()
S6_COMPONENT = 'MODE = "submit"\n\nK1_BUILD_ID = "2026-10-12T10:12:34+03:00/f0d87208a3de20a5ac936388bfd763799d6391833fea3a4f90ec6de14ef5051b"\n\nimport base64, faulthandler, hashlib, json, math, os, re, shutil, signal, subprocess, sys, tarfile, threading, time, traceback\n\nfrom pathlib import Path\n\nimport numpy as np\n\nimport pandas as pd\n\nKROOT = Path(os.environ.get("K1_KAGGLE_ROOT", "/kaggle"))\n\nINPUT, WORK = KROOT / "input", KROOT / "working"\n\nOUT = WORK / "k1"\n\nTMP = Path(os.environ.get("K1_TMP", "/tmp/k1"))\n\nCOMP = "rsna-knee-abnormality-detection"\n\nDS_SLUG = "rsna-knee-s6fall-weights"\n\nHELIOS_SLUG = "rsna-orthofoundation-fold0-epoch7-20261004"\n\nEXTRAS_FILE = "s6_extras.bin"\n\nLAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",\n       "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\n\nW_A, W_B = 0.0, 1.0\n\nA_SOURCE_SHA256 = "bcd864cbf349882e876959b49cb348cbb89b4c62fc204d4e4261049c5e3a79c5"\n\nA_PUBLIC_CSV_SHA256 = "faa20e95a1779658237252388407eaf6d835679dd43e4dae5bca18d95496bede"\n\nPUBLIC_TEST_IDS = ["1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191",\n                   "1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776",\n                   "1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402"]\n\nA_FALLBACK_MIN, A_FALLBACK_FRAC = 3, 0.01\n\nB_MODEL = {"file": "E_prior_s42_fall.pt", "source_file": "E_prior_s42_fall.pt", "sha256": "0c7297d58f061ab4e5d7c55046f51fa3e8934928046f0647a0c7854e98371c12", "run_config": "E_prior_s42_fall.run_config.json", "done": "E_prior_s42_fall.DONE.json", "arm": "E_prior", "expect_epochs": 16, "expect_train": 4263}\n\nANATOMY_FILE, ANATOMY_SHA256 = "anatomy.pt", "5904dd16eb9a4fd7a1caf90ac977f8fe2ef364908598c23c209db3fe7fd1563e"\n\nPAYLOAD_FILE, PAYLOAD_MAGIC = "k1_code_payload.bin", b"K1PAYLOAD1\\n"\n\nSOURCE_SHA = {                                                 # the 0.954 notebook\'s SOURCE_SHA = S6F0/s6common.SOURCE_SHA\n    "px_cache.py": "b13e38a7ad3bd496f8027e02209653e224b0926db4f0853eeb0d076a2d6e2632",\n    "compact_train.py": "14767c7a8f0daa986477511fe9a83aefe032a208ae978fabf15a5f896992ac80",\n    "compact_infer.py": "34f433fe72b732543813165c3397ec80cef01b0d49b25b7b22668714f0b0ef4a",\n    "probe_infer.py": "a6cc31448019c84be5868d0a2ff1d99719b23b0f86110efc03efff3ce6bd6980",\n    "knee_anchor_bank_v2.py": "e564807a129dfcd3c984348917436ee6083066a4d685340587ebd1d3ea14f309",\n    "side_common.py": "bbcc5d027834e6a6fd48c4a8cf228e25661b7522c792587c840fe4d7cd5c0566",\n}\n\nB_RECEIPT_SOURCES = ("px_cache.py", "compact_train.py", "compact_infer.py", "probe_infer.py", "knee_anchor_bank_v2.py")\n\nCODE_SHA = {                                                   # the 0.954 notebook\'s CODE_SHA (cloud_code/training pins)\n    "native_cache.py": "c53ce8985406b3b918d9e821d43518479e89d9f946daebd817f378e3949718b9",\n    "native_data.py": "f59afe205c8e193369fa3ccd22edcce8df3b293205fc48ddb1dd45169ee43b36",\n    "input_provenance.py": "b3a22500c7f976f6ee040239ada5cacecf2326861cb49af1c7d8cfdeeb55fcda",\n    "intensity_mapping.py": "19be2d8cbd6f8971c1288d7512a8cca41d143470c2ba6e9b9096fde035fa9d41",\n    "coronal_views.py": "841f216d2682ae2c0f8afd82fc902b9fa74924d7145b952069dea8b0bbd03c40",\n    "exact_duplicates.py": "51b38db21ce9c5fe2b6278a835983414756b61ac6af81aba380b44b41b075897",\n    "dicom_geometry.py": "12aa501f9704aef15b7a3bbd560951acb6d95b89b7de0141d6cf89fc3f051ab1",\n    "common.py": "8283fa1f8e7ef8cfacbba52df1715636be3c9ad688e39912814e7d5bcf57105f",\n}\n\nB_GEOMETRY = {"backbone": "convnext_small", "size": 320, "crop": 288, "views": "v4", "slices": 3, "phys_mm": 3.5,\n              "eval_stride": 1, "chunk": 192, "item_timeout": 900}\n\nB_AMP = "fp16"\n\nB_WORKERS = 2\n\nMIN_RHO_MEAN, MIN_RHO_FINDING, RHO_MIN_N = 0.60, 0.40, 100\n\nB_FAIL_MIN, B_FAIL_FRAC = 3, 0.0025\n\nA_TIMEOUT_S = 9000\n\nB_TIMEOUT_S = 14400\n\nPARITY_TIMEOUT_S = 3600\n\nGUARD_MAX_S = 480 * 60\n\nSETUP_SLACK_S = 30 * 60\n\nEND_SLACK_S = 10 * 60\n\nA_STALL_S = 30 * 60\n\nA_RETRY_MIN_S = 45 * 60\n\nEXIT_GRACE_S = 120\n\nB_DEADLINE_MARGIN_S = 20 * 60\n\nB_RETRY_MIN_S = 10 * 60\n\nGUARD_S = GUARD_MAX_S + (PARITY_TIMEOUT_S if MODE == "parity" else 0)\n\nassert A_TIMEOUT_S + B_TIMEOUT_S + SETUP_SLACK_S <= GUARD_MAX_S, "budgets exceed the hang guard"\n\nBACKSTOP_EXTRA_S = 10 * 60\n\nHEARTBEAT_S = 300\n\nPARITY_IDS = []\n\nT0 = time.time()\n\n_LOG = None\n\n_GUARD_FILE = None\n\n_CHILDREN = {}\n\n_GUARD_STOP = threading.Event()\n\n_RUN = {"nonce": os.urandom(16).hex(), "started_utc": time.strftime("%Y-%m-%dT%H:%M:%S+00:00", time.gmtime(T0)),\n        "kaggle_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE")}\n\n_RECEIPT = {"mode": MODE, "build_id": K1_BUILD_ID, "run": _RUN, "stages": {}, "gpu": {}}\n\nclass K1Error(RuntimeError):\n    pass\n\nclass K1Timeout(K1Error):\n    """a stage hit its time cap, or (A attempt 1) its log stalled"""\n\ndef log(*a):\n    msg = "%s +%6.0fs %s" % (time.strftime("%H:%M:%S", time.gmtime()), time.time() - T0, " ".join(str(x) for x in a))\n    print(msg, flush=True)\n    if _LOG is not None:\n        try:\n            _LOG.write(msg + "\\n"); _LOG.flush()\n        except Exception:\n            pass\n\ndef sha256_file(p):\n    h = hashlib.sha256()\n    with open(p, "rb") as f:\n        for b in iter(lambda: f.read(1 << 20), b""):\n            h.update(b)\n    return h.hexdigest()\n\ndef build_id_of(text):\n    """K1_BUILD_ID recomputed from a run.py text (= build_k1.build_id_of = compare_parity.build_id_of): \'<at>/<sha256(at + newline\n    + the text with the constant\'s value put back to the template placeholder)>\'; None if the text has no built id line."""\n    m = re.search(r\'^K1_BUILD_ID = "([^"/]+)/([0-9a-f]{64})"\', text, re.M)\n    if m is None:\n        return None\n    base = text[:m.start()] + \'K1_BUILD_ID = "__K1_\' + \'BUILD_ID__"\' + text[m.end():]     # split: the template holds one placeholder\n    return m.group(1) + "/" + hashlib.sha256((m.group(1) + "\\n" + base).encode()).hexdigest()\n\ndef self_identity():\n    """This execution\'s own source, when the runner exposes it (Kaggle runs a script as /kaggle/src/script.py; a notebook-style\n    wrapper may define no __file__): its path, sha256 and the build id recomputed from its text (derived_matches: that id equals\n    the baked K1_BUILD_ID).  Only a file holding this build\'s K1_BUILD_ID line counts.  Never raises; None when unavailable."""\n    out = {"path": None, "sha256": None, "derived_build_id": None, "derived_matches": None}\n    for cand in (globals().get("__file__"), sys.argv[0] if sys.argv else None):\n        try:\n            if not cand or not os.path.isfile(cand):\n                continue\n            text = Path(cand).read_text(errors="replace")\n            if (\'K1_BUILD_ID = "%s"\' % K1_BUILD_ID) not in text:\n                continue\n            out.update(path=str(cand), sha256=sha256_file(cand), derived_build_id=build_id_of(text))\n            out["derived_matches"] = out["derived_build_id"] == K1_BUILD_ID\n            break\n        except Exception as e:\n            out["error"] = repr(e)[:200]\n    return out\n\ndef save_json(path, obj):\n    path = Path(path); tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(obj, indent=1, default=str)); os.replace(tmp, path)\n\ndef tail(path, n=3000):\n    try:\n        return Path(path).read_text(errors="replace")[-n:]\n    except Exception as e:\n        return "(no log: %r)" % e\n\ndef remove_submission():\n    for p in (WORK / "submission.csv", WORK / ".submission.csv.tmp"):\n        try:\n            p.unlink()\n        except FileNotFoundError:\n            pass\n\ndef gpu_snapshot():\n    try:\n        r = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total,utilization.gpu", "--format=csv,noheader"],\n                           capture_output=True, text=True, timeout=20)\n        return [l.strip() for l in r.stdout.splitlines() if l.strip()] or [r.stderr.strip()[:200]]\n    except Exception as e:\n        return ["nvidia-smi failed: %r" % e]\n\ndef mem_available_mb():\n    try:\n        for line in open("/proc/meminfo"):\n            if line.startswith("MemAvailable:"):\n                return int(line.split()[1]) // 1024\n    except Exception:\n        pass\n    return -1\n\ndef group_rss_mb(pgid):\n    tot = 0\n    for d in os.listdir("/proc"):\n        if not d.isdigit():\n            continue\n        try:\n            st = open("/proc/%s/stat" % d).read()\n            f = st[st.rindex(")") + 2:].split()\n            if int(f[2]) == pgid:\n                tot += int(f[21]) * os.sysconf("SC_PAGE_SIZE")\n        except Exception:\n            continue\n    return tot // (1 << 20)\n\ndef kill_group(pgid):\n    try:\n        os.killpg(pgid, signal.SIGKILL)\n    except (ProcessLookupError, PermissionError):\n        pass\n    _CHILDREN.pop(pgid, None)\n\ndef kill_all_children():\n    for pgid in list(_CHILDREN):\n        kill_group(pgid)\n\ndef stage(name):\n    _RECEIPT["stages"][name] = {"start_s": round(time.time() - T0, 1)}\n    _RECEIPT["gpu"][name + ".start"] = gpu_snapshot()\n    log("[stage] %s start | gpu %s | MemAvailable %d MB" % (name, _RECEIPT["gpu"][name + ".start"], mem_available_mb()))\n\ndef stage_done(name, **info):\n    s = _RECEIPT["stages"][name]\n    s["seconds"] = round(time.time() - T0 - s["start_s"], 1)\n    s.update(info)\n    _RECEIPT["gpu"][name + ".end"] = gpu_snapshot()\n    log("[stage] %s done in %.0fs | gpu %s" % (name, s["seconds"], _RECEIPT["gpu"][name + ".end"]))\n\ndef _watchdog(cap_s):\n    deadline = T0 + cap_s\n    while True:\n        left = deadline - time.time()\n        if left <= 0:\n            break\n        if _GUARD_STOP.wait(min(left, 15)):\n            return\n    try:\n        n_children = len(_CHILDREN)\n        kill_all_children()                                  # cleanup first: a failing dump below must not skip it\n        remove_submission()\n        log("[guard] FIRED after %.0fs (cap %ds): killed %d child group(s), deleted submission.csv; dumping every thread"\n            % (time.time() - T0, cap_s, n_children))\n        for target in (_GUARD_FILE, sys.stdout):\n            try:\n                faulthandler.dump_traceback(file=target, all_threads=True)\n            except Exception as e:\n                log("[guard] dump to %r failed: %r" % (target, e))\n        try:\n            save_json(OUT / "k1_error.json", {"error": "hang guard fired", "cap_s": cap_s, "elapsed_s": round(time.time() - T0, 1),\n                                              "receipt": _RECEIPT})\n        except Exception:\n            pass\n        sys.stdout.flush()\n    finally:\n        os._exit(1)\n\ndef arm_guard(cap_s):\n    global _GUARD_FILE\n    _GUARD_FILE = open(OUT / "k1_hang_guard_stacks.txt", "w")\n    faulthandler.enable(file=_GUARD_FILE, all_threads=True)\n    faulthandler.dump_traceback_later(cap_s + BACKSTOP_EXTRA_S, repeat=False, file=_GUARD_FILE, exit=True)\n    threading.Thread(target=_watchdog, args=(cap_s,), name="k1-guard", daemon=True).start()\n    log("[guard] armed: watchdog %ds (kill children, delete submission.csv, _exit 1); faulthandler backstop %ds"\n        % (cap_s, cap_s + BACKSTOP_EXTRA_S))\n\ndef guard_left():\n    return T0 + GUARD_S - time.time()\n\ndef _size(path):\n    try:\n        return os.path.getsize(path)\n    except OSError:\n        return -1\n\ndef run_group(specs, timeout_s, what):\n    """Start every spec {name, cmd, env, cwd, log[, done_when, stall_s]} in its own session; poll until all exit, one fails, or\n    timeout_s passes.  A failure, a stall or the timeout kills every group (DataLoader / Pool children included).\n    done_when(): the child\'s outputs are complete; if it then has not exited within EXIT_GRACE_S (an interpreter-exit hang), its\n    group is killed and the run counts as a success.  stall_s: the log has not grown for that long = K1Timeout.\n    Returns [(name, rc, seconds, note)]."""\n    procs, t_start, last_hb, last_check = [], time.time(), time.time(), 0.0\n    try:\n        for s in specs:\n            fh = open(s["log"], "w")\n            p = subprocess.Popen(s["cmd"], env=s["env"], cwd=s.get("cwd"), stdout=fh, stderr=subprocess.STDOUT,\n                                 stdin=subprocess.DEVNULL, start_new_session=True)\n            _CHILDREN[p.pid] = s["name"]\n            procs.append({"s": s, "p": p, "fh": fh, "t1": time.time(), "size": -1, "changed": time.time(), "done_at": None})\n            log("[%s] started %s pid %d" % (what, s["name"], p.pid))\n        done = {}\n        while len(done) < len(procs):\n            now_ = time.time()\n            check = now_ - last_check >= 5\n            if check:\n                last_check = now_\n            for r in procs:\n                s, p = r["s"], r["p"]\n                if s["name"] in done:\n                    continue\n                rc, note = p.poll(), ""\n                if rc is None and check and s.get("done_when") is not None:\n                    try:\n                        complete = bool(s["done_when"]())\n                    except Exception:\n                        complete = False\n                    if complete and r["done_at"] is None:\n                        r["done_at"] = now_\n                    if r["done_at"] is not None and now_ - r["done_at"] > EXIT_GRACE_S and p.poll() is None:\n                        kill_group(p.pid)\n                        rc, note = 0, "outputs complete, no exit after %ds: group killed, accepted" % EXIT_GRACE_S\n                        log("[%s] %s: %s" % (what, s["name"], note))\n                if rc is not None:\n                    done[s["name"]] = (rc, round(time.time() - r["t1"], 1), note)\n                    r["fh"].close(); kill_group(p.pid)\n                    log("[%s] %s exited rc %d after %.0fs" % (what, s["name"], rc, time.time() - r["t1"]))\n                    if rc != 0:\n                        for r2 in procs:\n                            if r2["s"]["name"] not in done:\n                                kill_group(r2["p"].pid); log("[%s] killed %s (a sibling failed)" % (what, r2["s"]["name"]))\n                        raise K1Error("%s: %s exited %d; log tail:\\n%s" % (what, s["name"], rc, tail(s["log"])))\n                    continue\n                if check and s.get("stall_s"):\n                    sz = _size(s["log"])\n                    if sz != r["size"]:\n                        r["size"], r["changed"] = sz, now_\n                    elif now_ - r["changed"] > s["stall_s"]:\n                        for r2 in procs:\n                            if r2["s"]["name"] not in done:\n                                kill_group(r2["p"].pid)\n                        raise K1Timeout("%s: %s stalled: log unchanged for %ds (at %d B); log tail:\\n%s" % (\n                            what, s["name"], s["stall_s"], sz, tail(s["log"], 1500)))\n            if len(done) == len(procs):\n                break\n            if time.time() - t_start > timeout_s:\n                for r in procs:\n                    if r["s"]["name"] not in done:\n                        kill_group(r["p"].pid)\n                raise K1Timeout("%s: timeout after %ds; still running %s" % (\n                    what, timeout_s, [r["s"]["name"] for r in procs if r["s"]["name"] not in done]))\n            if time.time() - last_hb >= HEARTBEAT_S:\n                last_hb = time.time()\n                for r in procs:\n                    if r["s"]["name"] in done:\n                        continue\n                    prog = [l for l in tail(r["s"]["log"], 4000).splitlines() if l.startswith("{") or "[gpu]" in l or "[pre]" in l][-1:]\n                    log("[heartbeat] %s %s | %.0fs | rss %d MB | MemAvailable %d MB | gpu %s | %s" % (\n                        what, r["s"]["name"], time.time() - r["t1"], group_rss_mb(r["p"].pid), mem_available_mb(), gpu_snapshot(),\n                        prog[0][:300] if prog else "(no progress line yet)"))\n            time.sleep(2)\n        return [(r["s"]["name"],) + done[r["s"]["name"]] for r in procs]\n    finally:\n        for r in procs:\n            if r["p"].poll() is None:\n                kill_group(r["p"].pid)\n            try:\n                r["fh"].close()\n            except Exception:\n                pass\n\nPROBE = r\'\'\'\nimport json, sys, importlib.util\nout = {"python": sys.version.split()[0]}\nimport torch\nout.update(torch=torch.__version__, cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),\n           cuda_available=torch.cuda.is_available(), n_gpu=torch.cuda.device_count())\nif torch.cuda.is_available():\n    out["gpus"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]\n    out["capability"] = list(torch.cuda.get_device_capability(0))\n    out["bf16_supported"] = bool(torch.cuda.is_bf16_supported())\nfor m in ("numpy", "pandas", "timm", "pydicom", "cv2", "scipy", "safetensors"):\n    try:\n        out[m] = getattr(__import__(m), "__version__", "?")\n    except Exception as e:\n        out[m] = "MISSING " + type(e).__name__\nfor m in ("requests", "kagglesdk", "pylibjpeg", "gdcm"):\n    out["has_" + m] = importlib.util.find_spec(m) is not None\nprint("K1_ENV " + json.dumps(out), flush=True)\n\'\'\'\n\ndef probe_env():\n    r = subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True, timeout=300, stdin=subprocess.DEVNULL)\n    line = [l for l in r.stdout.splitlines() if l.startswith("K1_ENV ")]\n    if r.returncode != 0 or not line:\n        raise K1Error("environment probe failed rc %d: %s %s" % (r.returncode, r.stdout[-1500:], r.stderr[-1500:]))\n    return json.loads(line[0][len("K1_ENV "):])\n\ndef find_mount(slug, marker):\n    cands = [INPUT / slug, INPUT / "competitions" / slug]\n    cands += list(INPUT.glob("datasets/*/" + slug)) + list(INPUT.glob("datasets/*/" + slug + "/*"))\n    cands += list(INPUT.glob("*/" + slug)) + list(INPUT.glob(slug + "/*"))\n    found = []\n    for p in cands:\n        if (p / marker).is_file() and p.resolve() not in [f.resolve() for f in found]:\n            found.append(p)\n    if len(found) != 1:\n        raise K1Error("cannot resolve %s (marker %s): %s" % (slug, marker, found))\n    return found[0]\n\ndef safe_rel(name):\n    p = Path(name)\n    if p.is_absolute() or ".." in p.parts or not p.parts:\n        raise K1Error("unsafe payload member %r" % name)\n    return p\n\ndef extract_payload(path, target, members):\n    """The code payload = PAYLOAD_MAGIC + an uncompressed tar (opaque to Kaggle\'s archive handling); exact member set, sizes, sha."""\n    if target.exists():\n        shutil.rmtree(target)\n    with open(path, "rb") as f:\n        if f.read(len(PAYLOAD_MAGIC)) != PAYLOAD_MAGIC:\n            raise K1Error("payload magic mismatch")\n        with tarfile.open(fileobj=f, mode="r:") as tar:\n            ms = tar.getmembers()\n            if {m.name for m in ms} != set(members) or len(ms) != len(members):\n                raise K1Error("payload member set differs from MANIFEST")\n            for m in ms:\n                safe_rel(m.name)\n                if not m.isfile() or m.size != members[m.name]["bytes"]:\n                    raise K1Error("payload member %s: not a regular file or size differs" % m.name)\n            for m in ms:\n                dst = target / safe_rel(m.name)\n                dst.parent.mkdir(parents=True, exist_ok=True)\n                with tar.extractfile(m) as src, open(dst, "xb") as out:\n                    shutil.copyfileobj(src, out, 1 << 20)\n    for name, rec in members.items():\n        if sha256_file(target / name) != rec["sha256"]:\n            raise K1Error("payload member %s sha differs" % name)\n\nLAUNCHER = r\'\'\'"""K1 component-B launcher = S6F0/infer_s6f0.py with Kaggle paths.  Runs Heliosli\'s compact_infer.main() UNCHANGED.\nAs on the laptop: sys.path = [cloud_code, heliosli, ...] and the training.download_data stub (only the chunked-archive reader\nuses it; we read the DICOM directory), installed in the main process AND in every spawn worker (CStudies._setup).\nThree K1 guards, none of which changes a prediction on the laptop\'s path (each acts only where that path would fail):\n- distutils: the vendored MONAI 0.6.0 imports distutils.util.strtobool (Python >= 3.12 has no stdlib distutils; setuptools\'\n  shim provides it on the laptop).  Only if that import fails, a module holding the stdlib\'s strtobool is installed.\n- setup failure (review M1): compact_infer\'s _setup sets self.native before it loads the SKM-TEA V-Net, so a failed load left a\n  worker scoring every later study with field-centre views and no anchor, silently.  Here a failed _setup resets self.native\n  (the next study re-runs it; probe_infer.install_native is memoised, the one step that is not re-entrant) and re-raises (this\n  study becomes a fallback, retried); a scored item without an anchor is turned into a failed item (fallback, retried).\nK1_PX_DUMP (parity mode only): every item\'s pixel / meta hashes are appended to <dir>/px_<pid>.jsonl; the item is not changed."""\nimport hashlib, json, os, sys, types\nSRC = __SRC__\nCODE = __CODE__\nANCHOR_VIEWS = ("v3k", "v4")\n\n\ndef _strtobool(val):\n    """distutils.util.strtobool (CPython 3.11 / setuptools._distutils), unchanged semantics."""\n    val = val.lower()\n    if val in ("y", "yes", "t", "true", "on", "1"):\n        return True\n    elif val in ("n", "no", "f", "false", "off", "0"):\n        return False\n    else:\n        raise ValueError(f"invalid truth value {val!r}")\n\n\ndef install_distutils():\n    if "distutils.util" in sys.modules:\n        return\n    try:\n        import distutils.util  # noqa: F401  (setuptools\' shim on Python >= 3.12)\n    except Exception:\n        d, u = types.ModuleType("distutils"), types.ModuleType("distutils.util")\n        d.__path__ = []\n        u.strtobool = _strtobool\n        d.util = u\n        sys.modules["distutils"], sys.modules["distutils.util"] = d, u\n        os.environ["K1_DISTUTILS_SHIM"] = "1"\n\n\ndef install_paths():\n    sys.dont_write_bytecode = True\n    for p in (SRC, CODE):\n        if p not in sys.path:\n            sys.path.insert(0, p)\n    if "training.download_data" not in sys.modules:\n        m = types.ModuleType("training.download_data")\n        m.ChunkedArchive = None\n        sys.modules["training.download_data"] = m\n    install_distutils()\n\n\ndef memoise_install_native():\n    import probe_infer as PI\n    if getattr(PI, "_k1_memo", False):\n        return\n    orig, cache = PI.install_native, {}\n\n    def install_native(lean=True):\n        if lean not in cache:\n            cache[lean] = orig(lean)\n        return cache[lean]\n    PI.install_native = install_native\n    PI._k1_memo = True\n\n\ninstall_paths()\nimport compact_infer as CI\n\nassert os.path.dirname(os.path.abspath(CI.__file__)) == SRC, CI.__file__\nPX_DUMP = os.environ.get("K1_PX_DUMP")\n\n\nclass LocalCStudies(CI.CStudies):\n    def _setup(self):\n        install_paths()\n        memoise_install_native()\n        try:\n            super()._setup()\n            if self.views in ANCHOR_VIEWS and self.bank is None:\n                raise RuntimeError("K1: the SKM-TEA anchor bank did not load")\n        except BaseException:\n            self.native = None\n            raise\n\n    def __getitem__(self, i):\n        out = super().__getitem__(i)\n        if out.get("error") is None and self.views in ANCHOR_VIEWS and out.get("anchor") is None:\n            out = dict(out, error="K1: item built without an SKM-TEA anchor", images=out["images"][:0], meta=out["meta"][:0])\n        if PX_DUMP:\n            try:\n                img = out["images"].numpy()\n                meta = out["meta"].numpy()\n                rec = {"uid": out["uid"], "error": out["error"], "n": int(len(img)),\n                       "img_sha256": hashlib.sha256(img.tobytes()).hexdigest(),\n                       "meta_sha256": hashlib.sha256(meta.astype("float32").tobytes()).hexdigest(),\n                       "win_sha1_10": [hashlib.sha1(w.tobytes()).hexdigest()[:10] for w in img],\n                       "img_mean": float(img.mean()) if len(img) else None, "anchor": out.get("anchor"),\n                       "seconds": out.get("seconds")}\n            except Exception as exc:\n                rec = {"uid": out.get("uid"), "hash_error": repr(exc)[:200]}\n            with open(os.path.join(PX_DUMP, "px_%d.jsonl" % os.getpid()), "a", encoding="utf-8") as fh:\n                fh.write(json.dumps(rec) + "\\n")\n        return out\n\n\nCI.CStudies = LocalCStudies\n\nif __name__ == "__main__":\n    CI.main()\n\'\'\'\n\nB_PROBE = r\'\'\'\nimport importlib.util, json, os, sys, time\nlauncher, anatomy, model, backbone, crop, slices = sys.argv[1:7]\nt0 = time.time()\nout = {}\ntry:\n    import setuptools\n    out["setuptools"] = setuptools.__version__\nexcept Exception as e:\n    out["setuptools"] = "MISSING " + type(e).__name__\nout["has_distutils_hack"] = importlib.util.find_spec("_distutils_hack") is not None\nspec = importlib.util.spec_from_file_location("k1_b_launcher_probe", launcher)\nL = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(L)                      # the launcher\'s own path setup, stubs and distutils guard; imports compact_infer\nimport distutils.util, torch\nout["distutils"] = "K1 shim" if os.environ.get("K1_DISTUTILS_SHIM") else str(getattr(sys.modules["distutils"], "__file__", "?"))\nimport knee_anchor_bank_v2 as KB\nseg = KB.load_model(anatomy, "cpu")              # the vendored MONAI 0.6.0 V-Net + anatomy.pt, as every B worker loads it\nimport monai\nout["monai"] = [monai.__version__, os.path.relpath(monai.__file__, L.CODE)]\nwith torch.no_grad():\n    y = seg(torch.zeros(1, 1, 128, 128))\nout["vnet_out"] = list(y.shape)\nimport timm, compact_train as CT\nout["timm"] = timm.__version__\nsd = torch.load(model, map_location="cpu", weights_only=True)\nspatial = any(k == "sp_q" or k.endswith(".sp_q") for k in sd)\nnet = CT.Net(backbone, None, freeze_blocks=16, crop=int(crop), state_spec=None, in_chans=int(slices), spatial=spatial)\nnet.load_state_dict(sd, strict=True)            # = compact_infer.load_group for this model\nout.update(spatial=spatial, params=int(sum(p.numel() for p in net.parameters())), seconds=round(time.time() - t0, 1))\nprint("K1_BPROBE " + json.dumps(out), flush=True)\n\'\'\'\n\ndef probe_b_stack(I):\n    """B\'s own stack before anything is spent on A: the launcher (stubs, distutils guard), compact_infer import, the vendored\n    MONAI V-Net with anatomy.pt (+ one forward pass), compact_train.Net with the model\'s state dict (strict).  CPU only."""\n    env = dict(os.environ, CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS="2", PYTHONDONTWRITEBYTECODE="1", HF_HUB_OFFLINE="1")\n    env.pop("K1_DISTUTILS_SHIM", None)\n    g = B_GEOMETRY\n    r = subprocess.run([sys.executable, "-c", B_PROBE, str(I.launcher), str(I.anatomy), str(I.models / B_MODEL["file"]),\n                        g["backbone"], str(g["crop"]), str(g["slices"])],\n                       capture_output=True, text=True, timeout=900, stdin=subprocess.DEVNULL, env=env, cwd=str(TMP))\n    line = [l for l in r.stdout.splitlines() if l.startswith("K1_BPROBE ")]\n    if r.returncode != 0 or not line:\n        raise K1Error("component-B stack probe failed rc %d: %s %s" % (r.returncode, r.stdout[-1500:], r.stderr[-2500:]))\n    return json.loads(line[0][len("K1_BPROBE "):])\n\ndef write_launcher(src_dir, code_dir, path):\n    text = LAUNCHER.replace("__SRC__", repr(str(src_dir))).replace("__CODE__", repr(str(code_dir)))\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(text)\n    return path\n\nclass Inputs:\n    pass\n\ndef prepare_inputs():\n    I = Inputs()\n    I.comp = find_mount(COMP, "test.csv")\n    I.ds = find_mount(DS_SLUG, "checkpoint-manifest.json")\n    I.helios = find_mount(HELIOS_SLUG, ANATOMY_FILE)\n    log("competition root", I.comp, "| weights", I.ds, "| heliosli", I.helios)\n    man_path = I.ds / "code_manifest.json"\n    I.manifest_sha256 = sha256_file(man_path)\n    man = json.loads(man_path.read_text())\n    man["payload_members"] = man["members"]\n    if sha256_file(I.ds / B_MODEL["file"]) != B_MODEL["sha256"]:\n        raise K1Error("model sha differs from the kernel constant")\n    if sha256_file(I.helios / ANATOMY_FILE) != ANATOMY_SHA256:\n        raise K1Error("anatomy.pt differs from the S6 manifest anchor checkpoint")\n    cfg = json.loads((I.ds / B_MODEL["run_config"]).read_text())\n    want = {"size": B_GEOMETRY["size"], "crop": B_GEOMETRY["crop"], "views": B_GEOMETRY["views"], "slices": B_GEOMETRY["slices"],\n            "phys_mm": B_GEOMETRY["phys_mm"], "backbone": B_GEOMETRY["backbone"], "spatial_attn": True, "arms": B_MODEL["arm"]}\n    bad = {k: (cfg.get(k), v) for k, v in want.items() if cfg.get(k) != v}\n    if bad:\n        raise K1Error("run_config differs from the inference arguments (trained, inferred): %s" % bad)\n    done = json.loads((I.ds / B_MODEL["done"]).read_text())\n    if done.get("model_sha256") != B_MODEL["sha256"] or os.path.basename(done.get("model", "")) != B_MODEL.get("source_file", B_MODEL["file"]):\n        raise K1Error("DONE marker does not record this model")\n    for k in ("epochs", "train_studies"):\n        if B_MODEL.get("expect_" + k) is not None and done.get(k) != B_MODEL["expect_" + k]:\n            raise K1Error("DONE marker %s=%s, expected %s" % (k, done.get(k), B_MODEL["expect_" + k]))\n    I.code_root = TMP / "k1src"\n    # code = the scored run\'s exact file set: the extras from our dataset, everything else straight from Heliosli\'s; every\n    # file is checked against the scored sha256/bytes, so the tree is byte-identical to the one that scored 0.950\n    extras = {n: man["members"][n] for n in man["from_this_dataset"]}\n    extract_payload(I.ds / EXTRAS_FILE, I.code_root, extras)\n    for n, rec in man["members"].items():\n        if n in extras:\n            continue\n        src, dst = I.helios / safe_rel(n), I.code_root / safe_rel(n)\n        if not src.is_file() or src.stat().st_size != rec["bytes"]:\n            raise K1Error("heliosli file %s missing or size differs" % n)\n        dst.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copyfile(src, dst)\n        if sha256_file(dst) != rec["sha256"]:\n            raise K1Error("heliosli file %s sha differs" % n)\n    I.src = I.code_root / "heliosli"\n    I.code = I.code_root / "cloud_code"\n    for n, want_sha in SOURCE_SHA.items():\n        if sha256_file(I.src / n) != want_sha:\n            raise K1Error("heliosli/%s differs from SOURCE_SHA" % n)\n    for n, want_sha in CODE_SHA.items():\n        if sha256_file(I.code / "training" / n) != want_sha:\n            raise K1Error("cloud_code/training/%s differs from CODE_SHA" % n)\n    I.models = TMP / "k1models"\n    if I.models.exists():\n        shutil.rmtree(I.models)\n    I.models.mkdir(parents=True)\n    os.symlink(I.ds / B_MODEL["file"], I.models / B_MODEL["file"])        # the models folder holds exactly this one file\n    I.anatomy = I.helios / ANATOMY_FILE\n    I.launcher = write_launcher(I.src, I.code, TMP / "k1launch" / "k1_b_infer.py")\n    test = pd.read_csv(I.comp / "test.csv", dtype={"StudyInstanceUID": str})\n    I.test_ids = test["StudyInstanceUID"].astype(str).tolist()\n    if not I.test_ids or len(I.test_ids) != len(set(I.test_ids)):\n        raise K1Error("test.csv empty or has duplicate study ids")\n    ss = I.comp / "sample_submission.csv"\n    I.sub_cols = list(pd.read_csv(ss, nrows=1).columns) if ss.is_file() else ["StudyInstanceUID"] + LAB\n    if I.sub_cols[0] != "StudyInstanceUID" or sorted(I.sub_cols[1:]) != sorted(LAB):\n        raise K1Error("submission columns differ from the 12 findings: %s" % I.sub_cols)\n    I.cost = {}\n    for split in ("test", "train") if MODE == "parity" else ("test",):\n        sc = pd.read_csv(I.comp / (split + "_series.csv"), dtype=str, usecols=["StudyInstanceUID"])\n        I.cost.update(sc["StudyInstanceUID"].value_counts().to_dict())\n    log("inputs ok: %d test studies | MANIFEST %s | model %s %s... | %d payload files" % (\n        len(I.test_ids), I.manifest_sha256[:16], B_MODEL["file"], B_MODEL["sha256"][:16], len(man["payload_members"])))\n    _RECEIPT["inputs"] = {"comp": str(I.comp), "ds": str(I.ds), "manifest_sha256": I.manifest_sha256, "model": B_MODEL,\n                          "n_test": len(I.test_ids), "sub_cols": I.sub_cols}\n    return I\n\ndef a_fallback_cap(n):\n    return max(A_FALLBACK_MIN, math.ceil(A_FALLBACK_FRAC * n))\n\ndef b_cmd(I, split, assignment, out_dir, tmp_dir, deadline):\n    g = B_GEOMETRY\n    return [sys.executable, "-u", str(I.launcher), "--data", str(I.comp), "--split", split, "--code", str(I.code),\n            "--models", str(I.models), "--arm", B_MODEL["arm"], "--backbone", g["backbone"], "--size", str(g["size"]),\n            "--crop", str(g["crop"]), "--views", g["views"], "--slices", str(g["slices"]), "--phys-mm", str(g["phys_mm"]),\n            "--eval-stride", str(g["eval_stride"]), "--anchor-weights", str(I.anatomy), "--amp", B_AMP, "--chunk", str(g["chunk"]),\n            "--workers", str(B_WORKERS), "--item-timeout", str(g["item_timeout"]), "--assignment", str(assignment),\n            "--output", str(out_dir), "--tmp", str(tmp_dir), "--dump-logits", str(Path(out_dir) / "logits.json"),\n            "--deadline", "%.0f" % deadline]\n\ndef split_parts(ids, cost, n):\n    order = sorted(ids, key=lambda u: (-cost.get(u, 0), u))\n    parts = [order[i::n] for i in range(n)]\n    return [p[::2] + p[1::2] for p in parts if p]\n\ndef b_env(gi, px_dir=None):\n    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gi), OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1",\n               MALLOC_ARENA_MAX="2", PYTHONDONTWRITEBYTECODE="1", HF_HUB_OFFLINE="1")\n    env.pop("K1_PX_DUMP", None)\n    if px_dir is not None:\n        env["K1_PX_DUMP"] = str(px_dir)\n    return env\n\ndef collect_part(pdir, part, I):\n    rec = json.loads((pdir / "receipt.json").read_text())\n    if rec.get("status") != "PASS" or rec.get("studies") != len(part):\n        raise K1Error("%s: receipt status %s, studies %s (expected %d)" % (pdir.name, rec.get("status"), rec.get("studies"), len(part)))\n    if rec.get("amp") != B_AMP or rec.get("models") != [B_MODEL["file"]] or rec.get("spatial_models") != 1:\n        raise K1Error("%s: receipt amp %s models %s spatial %s" % (pdir.name, rec.get("amp"), rec.get("models"), rec.get("spatial_models")))\n    if rec.get("anchor_weights_sha256") != ANATOMY_SHA256:\n        raise K1Error("%s: anchor weights sha differs" % pdir.name)\n    if not isinstance(rec.get("timed_out"), bool):           # compact_infer always states it (True / False)\n        raise K1Error("%s: the receipt does not state timed_out (%r)" % (pdir.name, rec.get("timed_out")))\n    shas = rec.get("sha256")\n    if not isinstance(shas, dict) or not set(B_RECEIPT_SOURCES) <= set(shas):\n        raise K1Error("%s: the receipt\'s source shas are incomplete: %s" % (pdir.name, sorted(shas) if isinstance(shas, dict) else shas))\n    for n, s in shas.items():\n        if SOURCE_SHA.get(n) != s:\n            raise K1Error("%s: source %s sha differs in the receipt" % (pdir.name, n))\n    n_anchor = sum(int(v) for v in (rec.get("anchor_methods") or {}).values())\n    if n_anchor != rec.get("scored"):                  # every scored study must have had its SKM-TEA anchor record (review M1)\n        raise K1Error("%s: %s scored studies but %d anchor records %s" % (pdir.name, rec.get("scored"), n_anchor, rec.get("anchor_methods")))\n    logits = json.loads((pdir / "logits.json").read_text())\n    pc = pd.read_csv(pdir / "predictions.csv", dtype={"StudyInstanceUID": str})\n    if list(pc.columns) != ["StudyInstanceUID"] + LAB:\n        raise K1Error("%s: predictions.csv columns %s" % (pdir.name, list(pc.columns)))\n    pc = pc.set_index("StudyInstanceUID")\n    z = {}\n    for u in part:\n        if u not in logits:\n            continue\n        a = np.asarray(logits[u], np.float64)\n        if a.shape != (1, len(LAB)) or not np.isfinite(a).all():\n            raise K1Error("%s: logits for %s have shape %s or are non-finite" % (pdir.name, u, a.shape))\n        if u not in pc.index:\n            raise K1Error("%s: %s has logits but no predictions.csv row" % (pdir.name, u))\n        zm = a.mean(0)\n        p = 1.0 / (1.0 + np.exp(-zm))\n        d = float(np.abs(p - pc.loc[u, LAB].to_numpy(np.float64)).max())\n        if d >= 1e-5:\n            raise K1Error("%s: predictions.csv disagrees with the logits for %s (%.2e)" % (pdir.name, u, d))\n        z[u] = zm\n    extra = set(logits) - set(part)\n    if extra:\n        raise K1Error("%s: logits for studies outside the assignment: %s" % (pdir.name, sorted(extra)[:3]))\n    fallback = dict(rec.get("fallback") or {})\n    missing = [u for u in part if u not in z]\n    for u in missing:\n        fallback.setdefault(u, "no logits")\n    return z, fallback, rec\n\ndef progress_lines(logp):\n    out = []\n    for l in tail(logp, 10 ** 7).splitlines():\n        if l.startswith("{") and \'"done"\' in l:\n            try:\n                out.append(json.loads(l))\n            except Exception:\n                pass\n    return out\n\ndef b_fail_cap(n):\n    return max(B_FAIL_MIN, math.ceil(B_FAIL_FRAC * n)) if n >= RHO_MIN_N else 0\n\ndef b_done(pdir):\n    return (pdir / "receipt.json").is_file() and (pdir / "logits.json").is_file() and "WORKER_DONE" in tail(pdir / "log.txt", 3000)\n\ndef run_b(I, tag, ids, split, budget_s, px=False):\n    """B on ids over one process per GPU, one retry of failed studies.  Returns (P, Z, info, failed): failed = {uid: reason} of\n    studies still without a prediction (their P / Z rows are NaN); at most b_fail_cap(N) when tag == "B", else none.  A deadline\n    (first pass or retry) is always a HARD ERROR: a receipt whose timed_out is not False, or any fallback reason naming the\n    deadline.  px: every item\'s pixel / meta hashes are merged into <tag>/px_hashes.json.  info_sha256 (returned, so it reaches\n    k1_receipt.json) binds <tag>_info.json, which compare_parity reads for the amp and runtime checks."""\n    def deadline_named(fb):        # local, not module level: Astra\'s probe lifts run_b alone.  compact_infer\'s reason for a study\n        return [u for u, v in fb.items() if "deadline" in str(v).lower()]     # the deadline cut off: "not reached before the deadline"\n    root = OUT / tag\n    if root.exists():\n        shutil.rmtree(root)\n    root.mkdir(parents=True)\n    n_gpu = max(1, int(_RECEIPT["env"].get("n_gpu", 1)))\n    parts = split_parts(ids, I.cost, n_gpu)\n    t_start = time.time()\n    deadline = t_start + max(budget_s - B_DEADLINE_MARGIN_S, budget_s * 0.8)\n    specs, pdirs = [], []\n    for gi, part in enumerate(parts):\n        pdir = root / ("part%d" % gi)\n        pdir.mkdir()\n        (pdir / "ids.json").write_text(json.dumps(part))\n        px_dir = None\n        if px:\n            px_dir = pdir / "px"; px_dir.mkdir()\n        specs.append({"name": "%s.part%d" % (tag, gi), "cmd": b_cmd(I, split, pdir / "ids.json", pdir, TMP / ("%s_tmp%d" % (tag, gi)), deadline),\n                      "env": b_env(gi, px_dir), "cwd": str(TMP), "log": pdir / "log.txt", "done_when": (lambda d=pdir: b_done(d))})\n        pdirs.append(pdir)\n    (root / "argv_part0.json").write_text(json.dumps(specs[0]["cmd"]))\n    log("[%s] %d studies over %d GPU process(es) %s, budget %ds, compact_infer deadline +%ds" % (\n        tag, len(ids), len(parts), [len(p) for p in parts], budget_s, deadline - t_start))\n    runs = run_group(specs, budget_s, tag)\n    z, fallback, receipts = {}, {}, []\n    for pdir, part in zip(pdirs, parts):\n        zz, fb, rec = collect_part(pdir, part, I)\n        z.update(zz); fallback.update(fb)\n        receipts.append({k: rec.get(k) for k in ("studies", "scored", "seconds", "amp", "gpu", "torch", "timm", "stats", "timed_out",\n                                                 "anchor_methods", "per_study")})\n        receipts[-1]["progress"] = progress_lines(pdir / "log.txt")\n    timed = [i for i, rc in enumerate(receipts) if rc.get("timed_out") is not False]\n    if timed or deadline_named(fallback):\n        # the deadline came first: a retry cannot reach them either (review m4); no allowance ever covers a deadline.  A receipt\n        # that does not state timed_out False counts as one (collect_part already refuses it; this holds even without it)\n        raise K1Error("%s: compact_infer reached its deadline (budget %ds; timed_out not False in part(s) %s); %d studies have no "\n                      "prediction: %s" % (tag, budget_s, timed, len(fallback), dict(list(fallback.items())[:5])))\n    retry = None\n    if fallback:\n        left = t_start + budget_s - time.time()\n        log("[%s] %d fallback studies %s; %.0fs of budget left" % (tag, len(fallback), dict(list(fallback.items())[:5]), left))\n        if left < B_RETRY_MIN_S:\n            raise K1Error("%s: %d studies failed and no time is left for the retry: %s" % (tag, len(fallback), dict(list(fallback.items())[:10])))\n        rdir = root / "retry"\n        rdir.mkdir()\n        rids = sorted(fallback)\n        (rdir / "ids.json").write_text(json.dumps(rids))\n        rdeadline = time.time() + max(left - B_DEADLINE_MARGIN_S, left * 0.8)\n        if px:\n            (rdir / "px").mkdir()\n        rruns = run_group([{"name": tag + ".retry", "cmd": b_cmd(I, split, rdir / "ids.json", rdir, TMP / (tag + "_tmp_retry"), rdeadline),\n                            "env": b_env(0, (rdir / "px") if px else None), "cwd": str(TMP), "log": rdir / "log.txt",\n                            "done_when": lambda: b_done(rdir)}], left, tag)\n        zz, fb2, rec2 = collect_part(rdir, rids, I)\n        if rec2.get("timed_out") is not False or deadline_named(fb2):\n            # review C1: a retry that reaches its deadline is a HARD ERROR (PREREG section 5.1), never a capped B failure: the\n            # allowance below covers only studies that still fail after a retry that completed within its deadline\n            raise K1Error("%s: compact_infer reached its deadline in the retry (%.0fs left at its start; timed_out %r); %d of %d "\n                          "retried studies have no prediction: %s" % (tag, left, rec2.get("timed_out"), len(fb2), len(rids),\n                                                                       dict(list(fb2.items())[:5])))\n        z.update(zz)\n        retry = {"ids": rids, "first_reasons": fallback, "still_failing": fb2, "seconds": rec2.get("seconds"),\n                 "timed_out": rec2.get("timed_out"), "runs": rruns, "anchor_methods": rec2.get("anchor_methods")}\n        fallback = fb2\n    missing = [u for u in ids if u not in z]\n    failed = {u: fallback.get(u, "no logits") for u in missing}\n    cap = b_fail_cap(len(ids)) if tag == "B" else 0\n    if len(failed) > cap:\n        raise K1Error("%s: %d of %d studies have no B prediction after the retry (allowed %d): %s" % (\n            tag, len(failed), len(ids), cap, dict(list(failed.items())[:10])))\n    nan12 = np.full(len(LAB), np.nan)\n    Z = np.stack([z.get(u, nan12) for u in ids])\n    P = 1.0 / (1.0 + np.exp(-Z))\n    ok = np.array([u in z for u in ids])\n    if not np.isfinite(P[ok]).all():\n        raise K1Error("%s: non-finite B probabilities" % tag)\n    df = pd.DataFrame({"StudyInstanceUID": ids})\n    for j, l in enumerate(LAB):\n        df[l] = P[:, j]\n    for j, l in enumerate(LAB):\n        df["z|" + l] = Z[:, j]\n    df.to_csv(root / ("%s_probs.csv" % tag), index=False, float_format="%.17g")\n    info = {"seconds": round(time.time() - t_start, 1), "budget_s": round(budget_s), "parts": [len(p) for p in parts], "runs": runs,\n            "receipts": receipts, "retry": retry, "failed_after_retry": failed, "fail_cap": cap,\n            "probs_sha256": sha256_file(root / ("%s_probs.csv" % tag))}\n    if px:                                    # pixel evidence (review C3): one record per study; a retried study\'s retry record wins\n        recs, bad, clean = {}, [], lambda r: r.get("error") is None and not r.get("hash_error")\n        for f in sorted(root.glob("*/px/px_*.jsonl")):                     # part0, part1, ..., retry: sorted, so the retry is last\n            for k, line in enumerate(f.read_text(errors="replace").splitlines()):\n                try:\n                    r = json.loads(line)\n                except ValueError:\n                    r = None\n                if not isinstance(r, dict) or not isinstance(r.get("uid"), str):\n                    # a truncated line (a worker killed mid-write): diagnostic evidence never hard-fails the stage; the study is\n                    # then without a record unless another line covers it, and compare_parity FAILs missing evidence\n                    bad.append("%s:%d" % (f.relative_to(root), k + 1))\n                    continue\n                old = recs.get(r["uid"])\n                if old is None or clean(r) or not clean(old):    # a clean record is never replaced by an errored duplicate (a\n                    recs[r["uid"]] = r                           # loader restart can re-dump an item in another worker\'s file)\n        if bad:\n            log("[%s] px: %d unreadable line(s) skipped: %s" % (tag, len(bad), bad[:5]))\n        save_json(root / "px_hashes.json", recs)\n        info["px_hashes_sha256"] = sha256_file(root / "px_hashes.json")\n        info["px_studies"] = len(recs)\n        info["px_unreadable_lines"] = {"count": len(bad), "first": bad[:20]}\n    save_json(root / ("%s_info.json" % tag), info)\n    info["info_sha256"] = sha256_file(root / ("%s_info.json" % tag))   # the file holds everything above; the receipt binds it\n    log("[%s] ok: %d studies, %.0fs, retry %s, failed after retry %d (cap %d)" % (\n        tag, len(ids), info["seconds"], None if retry is None else len(retry["ids"]), len(failed), cap))\n    return P, Z, info, failed\n\ndef rank01(P):\n    """Per column: average-tie ranks 1..n divided by n, float64 (= scipy.stats.rankdata(method=\'average\') / n, gate_score.rank01)."""\n    P = np.asarray(P, np.float64)\n    n = P.shape[0]\n    out = np.empty(P.shape, np.float64)\n    for j in range(P.shape[1]):\n        v = P[:, j]\n        o = np.argsort(v, kind="mergesort")\n        s = v[o]\n        first = np.r_[True, s[1:] != s[:-1]]\n        starts = np.flatnonzero(first)\n        ends = np.r_[starts[1:], n] - 1\n        avg = (starts + ends) / 2.0 + 1.0\n        r = np.empty(n, np.float64)\n        r[o] = avg[np.cumsum(first) - 1]\n        out[:, j] = r / n\n    return out\n\ndef rank_blend(A, B, w_a=W_A):\n    return w_a * rank01(A) + (1.0 - w_a) * rank01(B)\n\ndef rho_per_finding(A, B):\n    """Spearman per finding; NaN for a constant column (exact test: np.std of equal floats can be 1e-17, not 0)."""\n    a, b = rank01(A), rank01(B)\n    return [float(np.corrcoef(a[:, j], b[:, j])[0, 1]) if np.ptp(a[:, j]) > 0 and np.ptp(b[:, j]) > 0 else float("nan")\n            for j in range(a.shape[1])]\n\ndef write_submission(ids, sub_cols, values):\n    sub = pd.DataFrame(np.asarray(values).astype(np.float32), columns=LAB)\n    sub.insert(0, "StudyInstanceUID", ids)\n    sub = sub[sub_cols]\n    if list(sub.columns) != sub_cols or sub["StudyInstanceUID"].tolist() != list(ids):\n        raise K1Error("submission column/row identity drift")\n    v = sub[LAB].to_numpy(np.float64)\n    if not np.isfinite(v).all() or (v < 0).any() or (v > 1).any():\n        raise K1Error("submission has non-finite or out-of-range values")\n    tmp = WORK / ".submission.csv.tmp"\n    sub.to_csv(tmp, index=False)\n    os.replace(tmp, WORK / "submission.csv")\n    return WORK / "submission.csv"\n\ndef blend_with_failures(A, B, failed_mask):\n    """= rank_blend(A, B) when nothing failed.  A study B failed on takes rank(A) (the 0.954 notebook\'s rule); B\'s ranks are taken\n    with those rows filled by the column mean of the scored rows (as Heliosli\'s controller fills a failed study)."""\n    failed_mask = np.asarray(failed_mask, bool)\n    if not failed_mask.any():\n        return rank_blend(A, B)\n    Bf = np.array(B, np.float64)\n    Bf[failed_mask] = np.nanmean(Bf[~failed_mask], axis=0)\n    rA = rank01(A)\n    out = W_A * rA + (1.0 - W_A) * rank01(Bf)\n    out[failed_mask] = rA[failed_mask]\n    return out\n\ndef sanity_floor(rho, n_ok):\n    """None if the floor holds (or N < RHO_MIN_N), else the reason.  A non-finite rho (a constant B column) fails it."""\n    rho = np.asarray(rho, np.float64)\n    if n_ok < RHO_MIN_N:\n        return None\n    if not np.isfinite(rho).all():\n        return "non-finite rank correlation for %s (a constant B column?)" % [LAB[j] for j in np.flatnonzero(~np.isfinite(rho))]\n    if rho.mean() < MIN_RHO_MEAN or rho.min() < MIN_RHO_FINDING:\n        return "rank corr(A, B) mean %.3f / min %.3f below %.2f / %.2f" % (rho.mean(), rho.min(), MIN_RHO_MEAN, MIN_RHO_FINDING)\n    return None\n\n\n_published_find_mount = find_mount\n\ndef mount_inventory():\n    entries = []\n    for base in (INPUT, INPUT/\'datasets\', INPUT/\'models\'):\n        if not base.is_dir():\n            continue\n        for p in sorted(base.iterdir()):\n            if p.is_dir():\n                entries.append(str(p))\n                if base != INPUT:\n                    entries.extend(str(q) for q in sorted(p.iterdir()) if q.is_dir())\n    return entries[:150]\n\ndef find_mount(slug, marker):\n    try:\n        return _published_find_mount(slug, marker)\n    except K1Error:\n        pass\n    found = []\n    blocked = {\'competitions\', \'train_series\', \'test_series\', \'study_cache\', \'__pycache__\'}\n    for current, dirs, files in os.walk(INPUT):\n        root = Path(current)\n        depth = len(root.relative_to(INPUT).parts)\n        if depth >= 6:\n            dirs[:] = []\n        else:\n            dirs[:] = [d for d in dirs if d not in blocked and d != COMP]\n        if marker in files:\n            resolved = root.resolve()\n            if resolved not in found:\n                found.append(resolved)\n    if len(found) == 1:\n        log(\'Resolved renamed/nested mount by marker:\', slug, found[0])\n        return found[0]\n    detail = {\'requested_slug\':slug, \'marker\':marker, \'candidates\':[str(p) for p in found],\n              \'attached_directories\':mount_inventory()}\n    OUT.mkdir(parents=True,exist_ok=True)\n    save_json(OUT/\'38_input_resolution.json\',detail)\n    raise K1Error(\'Missing or ambiguous input \'+slug+\'/\'+marker+\'. Candidates: \'+str(found)+\n                  \'. For anatomy.pt attach heliosli/rsna-orthofoundation-fold0-epoch7-20261004 Version 1 \'+\n                  \'(same anatomy/code dataset as successful 33). Merely attaching S6 weights is not sufficient. \'+\n                  \'See 38_input_resolution.json; do not rerun until attachments are corrected.\')\n\n\nCROP_TTA_PATCH = \'"""Compact-model inference worker (Kaggle or local): raw DICOM -> image scores for one list of studies.\\n\\nPath, identical to how the compact model was trained and evaluated:\\n  raw DICOM  --mainline training.native_cache.prepare (one study at a time)-->  native uint16 series\\n             --px_cache.StudySlices.build (the SAME file that built the pixel cache)-->  288 px slices\\n             --compact_train evaluation windows (every third slice as centre, both views, 3 adjacent slices)-->\\n             --centre crop 256, fold models of compact_train.Net, mean logit-->  12 probabilities\\nNothing is trained here.  No report, no label, no test-time fitting.  Failed or unreached studies are\\nlisted as fallbacks for the controller.\\n"""\\nimport argparse, contextlib, csv, hashlib, io, json, os, shutil, sys, time\\nfrom pathlib import Path\\nsys.dont_write_bytecode = True\\nHERE = Path(__file__).resolve().parent\\nimport numpy as np\\nimport torch\\n\\nUIDK = "StudyInstanceUID"\\nFORCED_SIDE = {}        # study uid -> "L" / "R": the native render uses this side instead of tags / sagittal IPP x\\n\\n\\nclass CStudies(torch.utils.data.Dataset):\\n    def __init__(self, data, split, uids, tmp, code, size, eval_stride=3, views="v1", slices=3, anchor_weights=None, phys_mm=0.0,\\n                 side_model=None, side_threshold=2.0, side_max_x=0.0, side_position_mm=20.0, extent_meta=False):\\n        self.data, self.split, self.uids = str(data), split, list(uids)\\n        # extent (M4) fold models read three more meta columns: series index, plane, slices in the series -- the same cache\\n        # columns (0, 2, 10) compact_train.Studies appends with extent_meta\\n        self.extent_meta = bool(extent_meta)\\n        self.tmp, self.code, self.size = str(tmp), str(code), int(size)\\n        self.eval_stride, self.views, self.slices = int(eval_stride), str(views), int(slices)\\n        self.phys_mm = float(phys_mm)\\n        self.anchor_weights = None if anchor_weights is None else str(anchor_weights)\\n        if self.views in ("v3k", "v4") and not self.anchor_weights:\\n            raise ValueError("views v3k need --anchor-weights (SKM-TEA checkpoint) to rebuild the knee anchors at inference")\\n        self.native = None\\n        # image side check (bundles that ship a side model): studies without one explicit DICOM laterality tag are\\n        # scored by the orientation classifier on their canonical coronal/axial slices; a confidently mirrored study is\\n        # re-rendered with the other side (native render, geometry, anchors and views all follow).  None = off.\\n        self.side_model = None if side_model is None else str(side_model)\\n        self.side_threshold, self.side_max_x = float(side_threshold), float(side_max_x)\\n        self.side_position_mm = float(side_position_mm)\\n        self.side_net = None\\n        self.source_directory = True     # False: read DICOMs from the chunked competition archive under <data>/download\\n\\n    def __len__(self):\\n        return len(self.uids)\\n\\n    def _setup(self):\\n        for p in (self.code, str(HERE)):\\n            if p not in sys.path:\\n                sys.path.insert(0, p)\\n        import probe_infer as PI\\n        self.PI = PI\\n        self.native = PI.install_native(True)\\n        import px_cache\\n        import compact_train\\n        self.builder = px_cache.StudySlices(self.tmp, [], self.size, self.views)   # (self.slices is the window depth K)\\n        self.window_rows = compact_train.window_rows          # the trainer\\\'s own window list (eval mode)\\n        self.bank = None\\n        if self.views in ("v3k", "v4"):\\n            # knee-centred views: per-series geometry from the DICOM headers (mainline geometry bank, the code that\\n            # built the training geometry) + SKM-TEA joint anchors (knee_anchor_bank_v2, the code of the training bank)\\n            import training.geometry_bank as GB\\n            import knee_anchor_bank_v2 as KB\\n            self.GB, self.KB = GB, KB\\n            dev = "cuda" if torch.cuda.is_available() else "cpu"\\n            if dev == "cpu":\\n                torch.set_num_threads(2)\\n            self.bank = KB.AnchorBank(KB.load_model(self.anchor_weights, dev), dev)\\n        import training.dicom_geometry as G\\n        if not getattr(G, "_forced_side_wrapped", False):\\n            import dataclasses\\n            orig = G.decide_study_side\\n\\n            def decide(study, tag_values, sagittal_x, _orig=orig):\\n                r = _orig(study, tag_values, sagittal_x)\\n                f = FORCED_SIDE.get(str(study))\\n                return r if f is None else dataclasses.replace(r, side=f, source="image_side_check")\\n            G.decide_study_side = decide\\n            G._forced_side_wrapped = True\\n        if self.side_model:\\n            import side_common as SC\\n            self.SC = SC\\n            sdev = "cuda" if torch.cuda.is_available() else "cpu"\\n            net = SC.build_side_net()\\n            net.load_state_dict(torch.load(self.side_model, map_location="cpu", weights_only=True), strict=True)\\n            self.side_net, self.side_dev = net.eval().to(sdev), sdev\\n\\n    def _prepare(self, uid, work):\\n        with contextlib.redirect_stdout(io.StringIO()):\\n            self.native.prepare(self.data, work, split=self.split, study_ids=[uid], source_directory=self.source_directory)\\n        if self.bank is not None or self.side_net is not None:\\n            with contextlib.redirect_stdout(io.StringIO()):\\n                self.GB.build(root=self.data, manifest=str(work / "manifest.json"), output=str(work / "geometry"),\\n                              split=self.split, source_directory=self.source_directory)\\n\\n    def _native_and_geometry(self, uid, work):\\n        """native render + geometry bank for one study; with a side model, the image side check (may re-render)."""\\n        self.PI.OneStudySource.current = uid\\n        self._prepare(uid, work)\\n        if self.side_net is None:\\n            return None\\n        key = hashlib.sha256(uid.encode()).hexdigest()[:24]\\n        with open(work / "studies" / (key + ".json"), encoding="utf-8") as fh:\\n            study = json.load(fh)\\n        sides = {s.get("side") for s in study["series"]}\\n        cur = next(iter(sides)) if len(sides) == 1 else None\\n        status = set()\\n        for s in study["series"]:\\n            gp = work / "geometry" / (hashlib.sha256((uid + "/" + s["SeriesInstanceUID"]).encode()).hexdigest()[:24] + ".json")\\n            if gp.exists():\\n                status.add(json.loads(gp.read_text(encoding="utf-8")).get("laterality_status"))\\n        sag = [float(np.median(s["positions"])) for s in study["series"] if s.get("plane") == "Sagittal" and s.get("positions")]\\n        x = float(np.median(sag)) if sag else None\\n        note = {"side": cur, "status": "|".join(sorted(str(v) for v in status)), "x": None if x is None else round(x, 1),\\n                "score": None, "flipped": False}\\n        if status == {"explicit_study_tags"} and cur in ("L", "R"):\\n            # one explicit DICOM laterality tag stands unless the knee lies clearly on the other side of the scanner\\n            # (10-07: 18 training studies had the tag contradicted by both the position and the images)\\n            pos = None if x is None or abs(x) < self.side_position_mm else ("L" if x >= 0 else "R")\\n            if pos is None or pos == cur:\\n                return note\\n            note["tag_contradicted_by_position"] = True\\n        if self.side_max_x > 0 and cur in ("L", "R") and x is not None and abs(x) >= self.side_max_x:\\n            return note                                   # far from the scanner centre: the sagittal IPP x rule holds\\n        ims = []\\n        for s in study["series"]:\\n            if s.get("plane") not in self.SC.SIDE_PLANES:\\n                continue\\n            p = Path(s["path"])\\n            if not p.exists():\\n                p = work / "series" / os.path.basename(str(s["path"]).replace(chr(92), "/"))\\n            if p.exists():\\n                ims.append(self.SC.side_inputs(np.load(p, mmap_mode="r")))\\n        ims = [v for v in ims if len(v)]\\n        if not ims:\\n            return note\\n        xb = torch.from_numpy(np.concatenate(ims)).float().div_(255.)[:, None]\\n        with torch.inference_mode():\\n            z = self.side_net(((xb - 0.45) / 0.25).to(self.side_dev)).float().squeeze(1)\\n        score = float(z.mean())\\n        note["score"] = round(score, 3)\\n        if score < -self.side_threshold:\\n            new = "R" if cur in (None, "L") else "L"\\n            FORCED_SIDE[uid] = new\\n            shutil.rmtree(work, ignore_errors=True)\\n            self._prepare(uid, work)\\n            with open(work / "studies" / (key + ".json"), encoding="utf-8") as fh:\\n                got = {s.get("side") for s in json.load(fh)["series"]}\\n            if got != {new}:\\n                raise ValueError("forced side %s not applied (%s)" % (new, got))\\n            note.update(flipped=True, new_side=new)\\n        return note\\n\\n    def rows(self, uid, forced_side=None):\\n        """training-cache override builder: the same native -> geometry -> anchors -> views path for one study, with an\\n        optional forced side; returns (StudySlices.build output, set of sides in the rendered study)."""\\n        if self.native is None:\\n            self._setup()\\n        key = hashlib.sha256(uid.encode()).hexdigest()[:24]\\n        work = Path(self.tmp) / ("r_" + key[:16])\\n        if forced_side is not None:\\n            FORCED_SIDE[uid] = forced_side\\n        try:\\n            self.PI.OneStudySource.current = uid\\n            with contextlib.redirect_stdout(io.StringIO()):\\n                self.native.prepare(self.data, work, split=self.split, study_ids=[uid], source_directory=self.source_directory)\\n                self.GB.build(root=self.data, manifest=str(work / "manifest.json"), output=str(work / "geometry"),\\n                              split=self.split, source_directory=self.source_directory)\\n            self.builder.cache = str(work)\\n            with open(work / "studies" / (key + ".json"), encoding="utf-8") as fh:\\n                study = json.load(fh)\\n            if self.bank is not None:\\n                rec = self.bank.record(uid, study, geo_dir=str(work / "geometry"))\\n                (work / "anchors").mkdir(exist_ok=True)\\n                with open(work / "anchors" / (self.KB.study_key(uid) + ".json"), "w", encoding="utf-8") as fh:\\n                    json.dump(rec, fh)\\n                self.builder.anchors = str(work / "anchors")\\n            return self.builder.build(uid), {s.get("side") for s in study["series"]}\\n        finally:\\n            FORCED_SIDE.pop(uid, None)\\n            shutil.rmtree(work, ignore_errors=True)\\n\\n    def __getitem__(self, i):\\n        uid = self.uids[i]\\n        t0 = time.perf_counter()\\n        work = Path(self.tmp) / ("s_" + hashlib.sha256(uid.encode()).hexdigest()[:16])\\n        t1 = t0\\n        try:\\n            if self.native is None:\\n                self._setup()\\n            side_note = self._native_and_geometry(uid, work)\\n            t1 = time.perf_counter()\\n            self.builder.cache = str(work)\\n            if self.bank is not None:\\n                key = self.KB.study_key(uid)\\n                with open(work / "studies" / (key + ".json"), encoding="utf-8") as fh:\\n                    study = json.load(fh)\\n                rec = self.bank.record(uid, study, geo_dir=str(work / "geometry"))\\n                (work / "anchors").mkdir(exist_ok=True)\\n                with open(work / "anchors" / (key + ".json"), "w", encoding="utf-8") as fh:\\n                    json.dump(rec, fh)\\n                self.builder.anchors = str(work / "anchors")\\n                anchor_note = {"method": rec["method"], "why": rec["why"][:4]}\\n            else:\\n                anchor_note = None\\n            t2 = time.perf_counter()\\n            item = self.builder.build(uid)\\n            px, meta = item["px"].numpy(), item["meta"].numpy()\\n            blocks = [{"n_slices": b["n_slices"], "views": [(0, v, st) for v, st in b["views"]]} for b in item["blocks"]]\\n            cand = self.window_rows(blocks, lambda f, r: meta[r], "eval", self.slices, self.eval_stride, phys_mm=self.phys_mm)\\n            if not cand:\\n                raise ValueError("no usable window")\\n            rows = np.array(cand, np.int64)                 # (file, centre row, K slice rows)\\n            img = np.stack([px[rows[:, 2 + d]] for d in range(self.slices)], 1)\\n            m = meta[rows[:, 1]]\\n            meta6 = np.stack([m[:, 1], m[:, 3], m[:, 4], m[:, 5], m[:, 6], m[:, 7]], 1).astype(np.float32)\\n            if self.extent_meta:\\n                meta6 = np.concatenate([meta6, np.stack([m[:, 0], m[:, 2], m[:, 10]], 1).astype(np.float32)], 1)\\n            out = {"uid": uid, "error": None, "images": torch.from_numpy(img), "meta": torch.from_numpy(meta6), "anchor": anchor_note,\\n                   "side": side_note}\\n        except Exception as exc:\\n            t2 = locals().get("t2", t1)\\n            out = {"uid": uid, "error": repr(exc)[:300], "images": torch.zeros((0, self.slices, self.size, self.size), dtype=torch.uint8),\\n                   "meta": torch.zeros((0, 9 if self.extent_meta else 6), dtype=torch.float32), "anchor": None}\\n        finally:\\n            shutil.rmtree(work, ignore_errors=True)\\n        out["seconds"] = [round(t1 - t0, 3), round(time.perf_counter() - t2, 3), round(t2 - t1, 3)]   # decode, render, anchor\\n        return out\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument("--data", required=True)\\n    ap.add_argument("--split", default="test")\\n    ap.add_argument("--code", required=True)\\n    ap.add_argument("--models", required=True, help="folder with <arm>_s<seed>_f<fold>.pt state dicts")\\n    ap.add_argument("--arm", required=True)\\n    ap.add_argument("--backbone", default="resnet34")\\n    ap.add_argument("--init", default=None, help="ofl_l backbone: the public OFL-L weights (safetensors) the fold models are rebuilt on")\\n    ap.add_argument("--freeze-blocks", type=int, default=16)\\n    ap.add_argument("--size", type=int, default=288)\\n    ap.add_argument("--crop", type=int, default=256)\\n    ap.add_argument("--views", default="v1", help="px_cache view set the fold models were trained with (v1 / v2)")\\n    ap.add_argument("--state-spec", default=None, help="state_spec json of the report-state construct the fold models were trained with")\\n    ap.add_argument("--slices", type=int, default=3, help="adjacent slices per window the fold models were trained with")\\n    ap.add_argument("--anchor-weights", default=None, help="views v3k: the SKM-TEA checkpoint (anatomy.pt of the code dataset) for the knee anchors")\\n    ap.add_argument("--phys-mm", type=float, default=0.0, help="window spacing in mm the fold models were trained with (compact_train --phys-mm)")\\n    ap.add_argument("--eval-stride", type=int, default=3, help="window centres: every k-th slice (3 = training-time validation default; 1 = every slice)")\\n    ap.add_argument("--assignment", default=None)\\n    ap.add_argument("--output", required=True)\\n    ap.add_argument("--tmp", default=None)\\n    ap.add_argument("--amp", default="auto", choices=["auto", "fp16", "bf16", "fp32"])\\n    ap.add_argument("--chunk", type=int, default=192)\\n    ap.add_argument("--workers", type=int, default=2)\\n    ap.add_argument("--prefetch", type=int, default=1)\\n    ap.add_argument("--item-timeout", type=float, default=900.)\\n    ap.add_argument("--deadline", type=float, default=0.)\\n    ap.add_argument("--dump-logits", default=None)\\n    ap.add_argument("--models2", default=None, help="second model group (another architecture), run on the same windows")\\n    ap.add_argument("--arm2", default=None, help="file prefix of the second group (default: --arm)")\\n    ap.add_argument("--backbone2", default=None)\\n    ap.add_argument("--w2", default=None, help="weight of group 2 per class: one number or 12 comma-separated (label order); "\\n                    "p = (1 - w) p_group1 + w p_group2")\\n    ap.add_argument("--side-model", default=None, help="orientation classifier (side_common.build_side_net state dict); off when absent")\\n    ap.add_argument("--side-threshold", type=float, default=2.0, help="flip a study only when its mean logit < -threshold")\\n    ap.add_argument("--side-max-x", type=float, default=0.0, help="> 0: skip the check for untagged studies this far (mm) from x = 0")\\n    ap.add_argument("--side-position-mm", type=float, default=20.0, help="a DICOM tag is re-checked only when the knee lies this far on the other side")\\n    a = ap.parse_args()\\n    out = Path(a.output)\\n    out.mkdir(parents=True, exist_ok=True)\\n    tmp = Path(a.tmp or (out / "tmp"))\\n    tmp.mkdir(parents=True, exist_ok=True)\\n    t_start = time.time()\\n    if a.assignment:\\n        uids = json.loads(Path(a.assignment).read_text(encoding="utf-8"))\\n    else:\\n        with open(Path(a.data) / (a.split + ".csv"), encoding="utf-8") as fh:\\n            uids = [r[UIDK] for r in csv.DictReader(fh)]\\n    if len(uids) != len(set(uids)):\\n        raise SystemExit("duplicate study ids")\\n\\n    dev = torch.device("cuda")\\n    for p in (a.code, str(HERE)):\\n        if p not in sys.path:\\n            sys.path.insert(0, p)\\n    import timm\\n    import compact_train as CT\\n    LABELS = CT.LABELS\\n    cap = torch.cuda.get_device_capability()[0]\\n    amp = a.amp if a.amp != "auto" else ("bf16" if cap >= 8 else "fp16")\\n    state_spec = json.load(open(a.state_spec, encoding="utf-8")) if a.state_spec else None\\n\\n    def load_group(folder, arm, backbone):\\n        fs = sorted(Path(folder).glob(arm + "_s*_f*.pt"))\\n        if not fs:\\n            raise SystemExit("no model files for arm %s in %s" % (arm, folder))\\n        ms, n_sp = [], 0\\n        for p in fs:\\n            sd = torch.load(p, map_location="cpu", weights_only=True)\\n            extent = any(k.split(".")[-1] in ("ext_u", "ext_view", "ext_gamma") for k in sd)   # M4 fold model (OA extent)\\n            if extent:\\n                N_EXTENT[0] += 1\\n            spatial = any(k == "sp_q" or k.endswith(".sp_q") for k in sd)       # P2 fold model (per-class spatial attention)\\n            n_sp += int(spatial)\\n            kw = {}\\n            if spatial:\\n                kw["spatial"] = True\\n            if extent:\\n                kw["extent"] = True\\n            m = CT.Net(backbone, a.init if backbone == CT.OFL_BACKBONE else None, freeze_blocks=a.freeze_blocks, crop=a.crop,\\n                       state_spec=state_spec, in_chans=a.slices, **kw)\\n            if backbone == CT.OFL_BACKBONE:\\n                m.load_deploy_state_dict(sd)          # fold model = public weights + trained tail/head\\n            else:\\n                m.load_state_dict(sd, strict=True)\\n            ms.append(m.eval().requires_grad_(False).to(dev).to(memory_format=torch.channels_last))\\n        return fs, ms, n_sp\\n\\n    N_EXTENT = [0]\\n    files, models, n_spatial = load_group(a.models, a.arm, a.backbone)\\n    files2, models2, n_spatial2, W2 = [], [], 0, None\\n    if a.models2:\\n        if not a.backbone2 or a.backbone2 == CT.OFL_BACKBONE or a.backbone == CT.OFL_BACKBONE:\\n            raise SystemExit("--models2 needs --backbone2 (timm backbones only, no OFL shared trunk)")\\n        files2, models2, n_spatial2 = load_group(a.models2, a.arm2 or a.arm, a.backbone2)\\n        w = [float(x) for x in (a.w2 or "0.5").split(",")]\\n        w = w * len(LABELS) if len(w) == 1 else w\\n        if len(w) != len(LABELS) or not all(0.0 <= x <= 1.0 for x in w):\\n            raise SystemExit("--w2: one weight or %d weights in [0, 1]" % len(LABELS))\\n        W2 = torch.tensor(w, dtype=torch.float64, device=dev)\\n    shared_trunk = a.backbone == CT.OFL_BACKBONE and len(models) > 1\\n    if shared_trunk:\\n        # all fold models share the frozen trunk bit-for-bit; keep one copy and run it once per window batch\\n        sd0 = models[0].enc.state_dict()\\n        for m in models[1:]:\\n            sdm = m.enc.state_dict()\\n            for k in models[0].enc.frozen_keys:\\n                if not torch.equal(sdm[k], sd0[k]):\\n                    raise SystemExit("fold models do not share the frozen trunk: " + k)\\n    print("MODEL_READY", json.dumps({"gpu": torch.cuda.get_device_name(), "amp": amp, "models": len(models),\\n                                     "models2": len(models2), "spatial_models": n_spatial + n_spatial2, "extent_models": N_EXTENT[0],\\n                                     "studies": len(uids), "seconds": round(time.time() - t_start, 1)}), flush=True)\\n    off = (a.size - a.crop) // 2\\n\\n    def run_model(m, img, meta, mask, mode):\\n        if mode == "fp32":\\n            return m(img, meta, mask, chunk=a.chunk).float()\\n        with torch.autocast("cuda", dtype=torch.float16 if mode == "fp16" else torch.bfloat16):\\n            return m(img, meta, mask, chunk=a.chunk).float()\\n\\n    def heads_from_tokens(m, tok, meta, mask):\\n        """compact_train.Net.head: encoder output -> class logits (the same code the trainer used, state construct included)."""\\n        return m.head(tok, meta, mask)\\n\\n    def run_shared(img, meta, mask, mode):\\n        """OFL fold models: frozen trunk once per window chunk, then every fold\\\'s trained tail + head."""\\n        x = img[mask]\\n        m0 = models[0]\\n        tails = [[] for _ in models]\\n        step = a.chunk or len(x)\\n        ctx = contextlib.nullcontext() if mode == "fp32" else torch.autocast("cuda", dtype=torch.float16 if mode == "fp16" else torch.bfloat16)\\n        with ctx:\\n            for s0 in range(0, len(x), step):\\n                xs = x[s0:s0 + step].float().div_(255.)\\n                xs = ((xs - m0.mean) / m0.std).contiguous(memory_format=torch.channels_last)\\n                tr = m0.enc.trunk(xs)\\n                for i, m in enumerate(models):\\n                    tails[i].append(m.enc.tail(tr))\\n            zs = [heads_from_tokens(m, torch.cat(tails[i]), meta, mask) for i, m in enumerate(models)]\\n        return torch.stack([z[0].float() for z in zs])\\n\\n    @torch.inference_mode()\\n    def score(images, meta):\\n        # Fixed same-size translations inside the random-crop range used in S6 training.\\n        if (a.size, a.crop) != (320, 288) or shared_trunk or models2:\\n            raise RuntimeError(\\\'41 crop TTA requires the pinned single S6 group at 320/288\\\')\\n        full = images.to(dev, non_blocking=True)\\n        meta = meta.to(dev)[None]\\n        mask = torch.ones((1, full.shape[0]), dtype=torch.bool, device=dev)\\n        crop_offsets = ((8, 8), (24, 24))\\n        per_crop = []\\n        for oy, ox in crop_offsets:\\n            img = full[None, :, :, oy:oy + a.crop, ox:ox + a.crop]\\n            zs = []\\n            for m in models:\\n                z = run_model(m, img, meta, mask, amp)\\n                if not torch.isfinite(z).all():\\n                    z = run_model(m, img, meta, mask, \\\'fp32\\\')\\n                zs.append(z[0])\\n            per_crop.append(torch.stack(zs))\\n        # Average complete study logits AFTER each crop\\\'s own spatial/MIL pooling.\\n        zs = torch.stack(per_crop).mean(0)\\n        return zs.mean(0), zs\\n\\n    fh = open(out / "predictions.csv", "w", encoding="utf-8", newline="")\\n    wr = csv.writer(fh)\\n    wr.writerow([UIDK] + LABELS)\\n    fallback, stats = {}, {"decode_s": 0., "render_s": 0., "gpu_s": 0., "windows": 0, "wait_s": 0., "rows": 0, "anchor_s": 0.}\\n    per_study, dump = [], {}\\n    anchor_methods = {}\\n    side_counts = {"checked": 0, "flipped": 0}\\n\\n    def handle(item):\\n        uid = item["uid"]\\n        if item["error"] or not len(item["images"]):\\n            fallback[uid] = item["error"] or "no usable window"\\n            return\\n        tg = time.perf_counter()\\n        try:\\n            zmean, zs = score(item["images"], item["meta"])\\n        except Exception as exc:\\n            fallback[uid] = "gpu: " + repr(exc)[:200]\\n            return\\n        if not torch.isfinite(zmean).all():\\n            fallback[uid] = "non-finite score"\\n            return\\n        p = torch.sigmoid(zmean).double().cpu().numpy()\\n        dt = time.perf_counter() - tg\\n        wr.writerow([uid] + ["%.6f" % float(x) for x in p])\\n        fh.flush()\\n        stats["rows"] += 1\\n        stats["decode_s"] += item["seconds"][0]\\n        stats["render_s"] += item["seconds"][1]\\n        stats["anchor_s"] += item["seconds"][2] if len(item["seconds"]) > 2 else 0.\\n        if item.get("side") and item["side"].get("score") is not None:\\n            side_counts["checked"] += 1\\n            side_counts["flipped"] += int(bool(item["side"].get("flipped")))\\n        if item.get("anchor"):\\n            anchor_methods[item["anchor"]["method"]] = anchor_methods.get(item["anchor"]["method"], 0) + 1\\n        stats["gpu_s"] += dt\\n        stats["windows"] += len(item["images"])\\n        per_study.append([uid, len(item["images"]), item["seconds"][0], item["seconds"][1], round(dt, 3)])\\n        if a.dump_logits:\\n            dump[uid] = zs.float().cpu().numpy().tolist()\\n\\n    pos = 0\\n    timed_out = False\\n    while pos < len(uids) and not timed_out:\\n        ds = CStudies(a.data, a.split, uids[pos:], tmp, a.code, a.size, a.eval_stride, a.views, a.slices, a.anchor_weights, a.phys_mm,\\n                      a.side_model, a.side_threshold, a.side_max_x, a.side_position_mm, extent_meta=N_EXTENT[0] > 0)\\n        loader = torch.utils.data.DataLoader(ds, batch_size=None, shuffle=False, num_workers=a.workers,\\n                                             prefetch_factor=a.prefetch if a.workers else None, persistent_workers=False,\\n                                             multiprocessing_context="spawn" if a.workers else None,\\n                                             timeout=a.item_timeout if a.workers else 0)\\n        try:\\n            mark = time.perf_counter()\\n            for item in loader:\\n                stats["wait_s"] += time.perf_counter() - mark\\n                handle(item)\\n                pos += 1\\n                if pos % 20 == 0 or pos == len(uids) or pos <= 3:\\n                    el = time.time() - t_start\\n                    print(json.dumps({"done": pos, "of": len(uids), "scored": stats["rows"], "fallback": len(fallback),\\n                                      "elapsed_s": round(el, 1), "s_per_study": round(el / max(pos, 1), 2),\\n                                      "decode_s": round(stats["decode_s"], 1), "render_s": round(stats["render_s"], 1),\\n                                      "gpu_s": round(stats["gpu_s"], 1), "anchor_s": round(stats["anchor_s"], 1),\\n                                      "anchors": anchor_methods, "windows": stats["windows"]}), flush=True)\\n                if a.deadline and time.time() > a.deadline:\\n                    timed_out = True\\n                    break\\n                mark = time.perf_counter()\\n        except Exception as exc:\\n            if pos < len(uids):\\n                fallback[uids[pos]] = "loader: " + repr(exc)[:200]\\n            pos += 1\\n        del loader\\n    for u in uids[pos:]:\\n        fallback[u] = "not reached before the deadline"\\n    fh.close()\\n    shutil.rmtree(tmp, ignore_errors=True)\\n    if a.dump_logits:\\n        Path(a.dump_logits).write_text(json.dumps(dump), encoding="utf-8")\\n    receipt = {"status": "PASS", "arm": a.arm, "models": [p.name for p in files], "backbone": a.backbone, "shared_trunk": bool(shared_trunk), "init": a.init,\\n               "spatial_models": n_spatial, "extent_models": N_EXTENT[0], "group2": None if not models2 else {"models": [p.name for p in files2], "backbone": a.backbone2,\\n                                                                                   "spatial_models": n_spatial2, "w2": W2.tolist()},\\n               "studies": len(uids), "scored": stats["rows"], "fallback": fallback, "timed_out": timed_out, "amp": amp,\\n               "gpu": torch.cuda.get_device_name(), "torch": torch.__version__, "timm": timm.__version__,\\n               "seconds": round(time.time() - t_start, 1), "stats": {k: round(v, 1) for k, v in stats.items()},\\n               "per_study": per_study[:50], "anchor_methods": anchor_methods, "views": a.views, "phys_mm": a.phys_mm,\\n               "side_check": None if not a.side_model else dict(side_counts, threshold=a.side_threshold, max_x=a.side_max_x, position_mm=a.side_position_mm,\\n                                                                  model_sha256=hashlib.sha256(Path(a.side_model).read_bytes()).hexdigest()),\\n               "anchor_weights_sha256": (hashlib.sha256(Path(a.anchor_weights).read_bytes()).hexdigest() if a.anchor_weights else None),\\n               "sha256": {n: hashlib.sha256((HERE / n).read_bytes()).hexdigest()\\n                          for n in ("px_cache.py", "compact_train.py", "compact_infer.py", "probe_infer.py")\\n                          + (("side_common.py",) if a.side_model else ())\\n                          + (("knee_anchor_bank_v2.py",) if a.views in ("v3k", "v4") else ())},\\n               "model": "compact fine-tuned image model on the pixel cache"}\\n    receipt[\\\'crop_tta\\\'] = {\\\'offsets\\\': [[8,8],[24,24]], \\\'crop\\\':288, \\\'cache_size\\\':320, \\\'combine\\\':\\\'equal study-logit mean\\\', \\\'passes\\\':2}\\n    (out / "receipt.json").write_text(json.dumps(receipt, indent=1), encoding="utf-8")\\n    print("WORKER_DONE", json.dumps({k: receipt[k] for k in ("studies", "scored", "timed_out", "seconds", "amp")}),\\n          "fallback", len(fallback), flush=True)\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\'\nCROP_TTA_PATCH_SHA = \'7da30ee45aa116e3f246fc5c58ab0d25c08a03d62a386618d75a220c7931c446\'\n\n_41_original_prepare_inputs = prepare_inputs\n\ndef prepare_inputs():\n    # First verify the entire ORIGINAL staged code against the pinned public manifest.\n    I = _41_original_prepare_inputs()\n    path = I.src/\'compact_infer.py\'\n    old_sha = sha256_file(path)\n    if old_sha != \'34f433fe72b732543813165c3397ec80cef01b0d49b25b7b22668714f0b0ef4a\':\n        raise K1Error(\'Original S6 worker identity differs before41 patch\')\n    path.write_bytes(CROP_TTA_PATCH.encode(\'utf-8\'))\n    if sha256_file(path) != CROP_TTA_PATCH_SHA:\n        raise K1Error(\'41 crop worker write identity mismatch\')\n    # Explicit new experiment identity, not a bypass of the original release guard.\n    SOURCE_SHA[\'compact_infer.py\'] = CROP_TTA_PATCH_SHA\n    _RECEIPT[\'inputs\'][\'experimental_worker\'] = {\n        \'original_sha256\':old_sha,\'patched_sha256\':CROP_TTA_PATCH_SHA,\n        \'crop_tta_offsets\':[[8,8],[24,24]],\'passes\':2,\'combine\':\'equal study-logit mean\'}\n    save_json(OUT/\'42_worker_patch_audit.json\',_RECEIPT[\'inputs\'][\'experimental_worker\'])\n    return I\n\n_41_original_collect_part = collect_part\n\ndef collect_part(pdir, part, I):\n    z,fb,rec = _41_original_collect_part(pdir,part,I)\n    expected={\'offsets\':[[8,8],[24,24]],\'crop\':288,\'cache_size\':320,\n              \'combine\':\'equal study-logit mean\',\'passes\':2}\n    if rec.get(\'crop_tta\') != expected:\n        raise K1Error(\'41 worker did not declare the prescribed crop TTA\')\n    return z,fb,rec\n'
S6_SCRIPT = "\nimport importlib.util, json, os, time, traceback\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nOUT=Path('/kaggle/working/rsna_43B')\nspec=importlib.util.spec_from_file_location('s6_39',OUT/'s6_component.py')\nS6=importlib.util.module_from_spec(spec)\nspec.loader.exec_module(S6)\nS6.OUT=OUT/'s6_outputs'\nS6.OUT.mkdir(parents=True,exist_ok=True)\nS6.TMP=Path('/tmp/rsna_43B_s6')\nS6.TMP.mkdir(parents=True,exist_ok=True)\nS6.B_GEOMETRY=dict(S6.B_GEOMETRY,chunk=96)\nS6.B_WORKERS=1\ntry:\n    ds=S6.find_mount(S6.DS_SLUG,'code_manifest.json')\n    if S6.sha256_file(ds/'code_manifest.json')!='b66c574b5c0c13ccaa49d1219d41a8781431166186ef83e78cab6a9ba6b02d22':\n        raise RuntimeError('S6 manifest changed; attach Version5')\n    I=S6.prepare_inputs()\n    env=S6.probe_env()\n    if not env.get('cuda_available'):raise RuntimeError('CUDA required')\n    S6._RECEIPT['env']=env\n    probe=S6.probe_b_stack(I)\n    budget=min(25200,float(os.environ['RSNA39_DEADLINE'])-time.time()-600)\n    if budget<1200:raise RuntimeError('Insufficient standalone S6 budget')\n    p,z,info,failed=S6.run_b(I,'B',I.test_ids,'test',budget)\n    if failed:raise RuntimeError('Standalone comparison requires every S6 row scored; failed='+repr(failed))\n    pd.DataFrame(p,index=pd.Index(I.test_ids,name='StudyInstanceUID'),columns=S6.LAB).to_csv(OUT/'39_s6_probabilities.csv')\n    S6.save_json(OUT/'39_s6_receipt.json',{'status':'COMPLETE','inputs':S6._RECEIPT['inputs'],\n        'environment':env,'probe':probe,'run':info,'failed':failed,'model_fallback_to_raptor':False})\nexcept BaseException as exc:\n    S6.kill_all_children()\n    S6.save_json(OUT/'39_s6_error.json',{'error':repr(exc),'traceback':traceback.format_exc()})\n    raise\n"
VISIBLE_REF = [{'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191', 'ACL': '0.04315763977901502', 'MCL': '0.08821125116787842', 'Medial Meniscus': '0.9514312132612571', 'Lateral Meniscus': '0.051314057643339885', 'Medial OA': '0.5022758460826032', 'Lateral OA': '0.10990827342974287', 'PF OA': '0.18623147792171268', 'Effusion': '0.5725009533048611', 'Synovitis': '0.20269653075705538', "Baker's": '0.9545977216345588', 'Contusion': '0.05522114959753875', 'Fracture': '0.08523676911141621'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776', 'ACL': '0.19050341779306612', 'MCL': '0.13614737565715615', 'Medial Meniscus': '0.8830431017904165', 'Lateral Meniscus': '0.0578383159937494', 'Medial OA': '0.6461985745964476', 'Lateral OA': '0.6089607761583417', 'PF OA': '0.9284630407021515', 'Effusion': '0.2992807328698879', 'Synovitis': '0.31335110023886337', "Baker's": '0.19363467669622755', 'Contusion': '0.36624419886827797', 'Fracture': '0.13592316360427587'}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402', 'ACL': '0.09430224981100319', 'MCL': '0.06574939871194475', 'Medial Meniscus': '0.20326437584042997', 'Lateral Meniscus': '0.09892122943301729', 'Medial OA': '0.8915136183372454', 'Lateral OA': '0.5214554335198262', 'PF OA': '0.9312781990509439', 'Effusion': '0.2294750506188816', 'Synovitis': '0.1427420518710936', "Baker's": '0.11496204511009224', 'Contusion': '0.07486148571236029', 'Fracture': '0.0782252657987691'}]


def make_standalone_submission(raw,ids,labels):
    if list(raw.columns)!=['StudyInstanceUID']+labels:
        raise RuntimeError('S6 target order mismatch')
    if not raw.StudyInstanceUID.is_unique or set(raw.StudyInstanceUID)!=set(ids):
        raise RuntimeError('S6 UID coverage mismatch')
    p=raw.set_index('StudyInstanceUID').loc[ids,labels]
    vals=p.to_numpy(float)
    if not np.isfinite(vals).all() or (vals<0).any() or (vals>1).any():
        raise RuntimeError('Invalid/missing standalone probabilities; no fallback is allowed')
    # AUC is rank-based. Use exactly the same monotone output transform in A/B.
    result=p.rank(method='average',pct=True).astype(np.float64)
    return result.reset_index()

clear_partial_submission()
try:
    (OUT/'s6_component.py').write_bytes(S6_COMPONENT.encode('utf-8'))
    spec=importlib.util.spec_from_file_location('s6_standalone_preflight',OUT/'s6_component.py')
    M=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(M)
    M.OUT=OUT
    comp=M.find_mount(M.COMP,'test.csv')
    anatomy=M.find_mount(M.HELIOS_SLUG,'anatomy.pt')
    ds=M.find_mount(M.DS_SLUG,'code_manifest.json')
    if M.sha256_file(ds/'code_manifest.json')!='b66c574b5c0c13ccaa49d1219d41a8781431166186ef83e78cab6a9ba6b02d22':
        raise RuntimeError('Attach S6 weights Version5; source manifest differs')
    if M.sha256_file(anatomy/'anatomy.pt')!='5904dd16eb9a4fd7a1caf90ac977f8fe2ef364908598c23c209db3fe7fd1563e':
        raise RuntimeError('Attach exact Heliosli anatomy/code Version1')
    model=ds/M.B_MODEL['file']
    if not model.is_file() or M.sha256_file(model)!=M.B_MODEL['sha256']:
        raise RuntimeError('Pinned S6 checkpoint missing/changed')
    ids=pd.read_csv(comp/'test.csv',dtype={'StudyInstanceUID':str}).StudyInstanceUID.tolist()
    if not ids or len(set(ids))!=len(ids):raise RuntimeError('Invalid test study IDs')
    labels=list(M.LAB)
    columns=list(pd.read_csv(comp/'sample_submission.csv',nrows=0).columns)
    if columns!=['StudyInstanceUID']+labels:raise RuntimeError('Unexpected submission column order')
    REPORT['test_rows']=len(ids)
    REPORT['input_directories']=M.mount_inventory()
    REPORT['component_sha256']=hashlib.sha256(S6_COMPONENT.encode()).hexdigest()
    save_report()
    run_stage('s6',S6_SCRIPT,7.75*3600)
    receipt=json.loads((OUT/'39_s6_receipt.json').read_text())
    if receipt.get('status')!='COMPLETE' or receipt.get('failed'):
        raise RuntimeError('S6 incomplete; no standalone fallback allowed')
    raw=pd.read_csv(OUT/'39_s6_probabilities.csv',dtype={'StudyInstanceUID':str})
    result=make_standalone_submission(raw,ids,labels)
    raw.to_csv(OUT/(EXPERIMENT+'_probabilities.csv'),index=False,float_format='%.17g')
    if set(ids)=={r['StudyInstanceUID'] for r in VISIBLE_REF}:
        ref=pd.DataFrame(VISIBLE_REF).set_index('StudyInstanceUID').loc[ids,labels].to_numpy(float)
        actual=raw.set_index('StudyInstanceUID').loc[ids,labels].to_numpy(float)
        delta=float(np.max(np.abs(ref-actual)))
        REPORT['visible_vs38_max_abs_probability_delta']=delta
        if REPORT['mode']=='centre' and (not np.isfinite(delta) or delta>2e-4):
            raise RuntimeError('Centre-only S6 does not reproduce successful38 visible probabilities')
        if REPORT['mode']!='centre':REPORT['visible_difference_expected']=True
    REPORT['s6_receipt']=receipt
    REPORT['warnings']=[]
    if time.time()>DEADLINE:raise RuntimeError('Global deadline exceeded')
    pending=OUT/'submission.pending.csv'
    result.to_csv(pending,index=False,float_format='%.17g')
    final=pd.read_csv(pending,dtype={'StudyInstanceUID':str})
    if list(final.columns)!=columns or final.StudyInstanceUID.tolist()!=ids:
        raise RuntimeError('Final CSV row/column mismatch')
    values=final[labels].to_numpy(float)
    if not np.isfinite(values).all() or (values<0).any() or (values>1).any():
        raise RuntimeError('Invalid final scores')
    REPORT.update(status='COMPLETE',rows=len(ids),submission_sha256=hashlib.sha256(pending.read_bytes()).hexdigest())
    save_report()
    os.replace(pending,WORK/'submission.csv')
    print(EXPERIMENT+' COMPLETE',json.dumps(REPORT),flush=True)
    display(result.head())
except BaseException as exc:
    clear_partial_submission()
    REPORT.update(status='FAILED',error=repr(exc),traceback=traceback.format_exc())
    save_report()
    raise
